# CEO Turnover and Executive Pay Dispersion in Europe
**Data & AI in Economics | TU Dortmund**

This notebook develops an analysis-ready panel for studying whether CEO turnover changes CEO compensation and pay dispersion in European firms. It begins with the STOXX Europe 600 universe, retrieves and validates BoardEx and Compustat data, identifies firm-level CEO spells and turnover events, matches CEOs to remuneration records, and combines the resulting CEO pay panel with firm fundamentals, market capitalization, executive characteristics, sector, and country information.

> **Team size:** 2 students  
> **Deliverable:** Jupyter Notebook, extended from proposal to final analysis


<a id="toc"></a>
## Table of Contents

1. [Team](#team)
2. [Research Question](#research-question)
3. [Data Sources and Variables](#data-sources-and-variables)
4. [Setup and Helper Functions](#helper-functions)
5. [Data Collection and Profiling](#data-collection)
   - [STOXX 600 Universe](#stoxx-600-universe)
   - [BoardEx Employment Records](#boardex-employment-records)
   - [BoardEx Executive Details](#boardex-executive-details)
   - [BoardEx Remuneration Records](#boardex-remuneration-records)
   - [Compustat Fundamentals](#compustat-fundamentals)
   - [Compustat Monthly Market Prices](#compustat-market-prices)
   - [Compustat Daily Exchange Rates](#compustat-exchange-rates)
6. [CEO Turnover and Pay Panel](#ceo-turnover-and-pay-panel)
   - [CEO Turnover Events](#ceo-turnover-events)
   - [Remuneration Cleaning and EUR Conversion](#remuneration-currency-conversion)
   - [CEO Remuneration Matching](#ceo-remuneration-matching)
   - [Firm-Year CEO Pay Panel](#firm-year-ceo-pay-panel)
7. [Market Data and Market Capitalization](#market-data-and-market-capitalization)
   - [Convert Annual Fundamentals to EUR](#fundamentals-eur-conversion)
   - [Select One Exchange per Firm](#market-exchange-filter)
   - [Convert Market Prices and Construct Market Cap](#market-eur-merge)
8. [Final Analysis Panel](#final-analysis-panel)
   - [Merge Analysis Inputs](#construct-final-panel)
   - [CPI Adjustment and Real-EUR Variables](#cpi-adjustment)
   - [Winsorization of Pay](#winsorization)
   - [Save Clean Panel](#save-panel)
9. [Data Visualizations](#data-visualizations)
   - [Target Distribution](#target-distribution)
   - [Annual Pay Trend and Coverage](#annual-pay-trend)
   - [Pay Around CEO Turnover](#turnover-event-visualization)
   - [Exploratory Relationships](#exploratory-relationships)
   - [Grouped Pay Comparisons](#grouped-pay-comparisons)
10. [Planned Methods](#planned-methods)
11. [Evaluation Strategy](#evaluation-strategy)
12. [Work Plan](#work-plan)
13. [Results and Discussion](#results-and-discussion)
   - [Causal Inference](#causal-inference-results)

**Reading guide:** run the notebook from top to bottom. The main dataset produced by the preparation workflow is `df_panel_final`; earlier tables are diagnostic checks that explain how each input contributes to that final panel.


<a id="team"></a>
## 1. Team


| Role | Name | Student ID |
|------|------|------------|
| Lead |Achmad Rizky Akbar| |
| Member | Kajetan Zduńczyk| |
| Member *(optional)* | | |


<a id="research-question"></a>
## 2. Mission Title & Research Question


**Title:** *CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes*

**Research question:** *Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

**Why it matters:** *By focusing on leadership changes observed in BoardEx, we can estimate how compensation responds to a governance shock without hand‑collecting policy data. This helps investors and regulators understand whether turnover acts as a disciplining mechanism on executive pay and internal pay gaps.*

<a id="data-sources-and-variables"></a>
## 3. Data Sources and Variables


**Source(s):**  

1. **BoardEx - Individual Profile Employment**

    Includes data about Individual Profile Employment for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-employment/

2. **BoardEx - Individual Profile Details**

    Includes data about Individual Profile Details for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-details/

3. **Company Profile Details - BoardEx (Wharton Data Research Services)**

    Company Profile Details includes data such as location, market cap, and sector.
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/company-profile/company-profile-details/

4. **Fundamentals Annual - Compustat Global (Wharton Data Research Services)**

    Provides fundamental annual company information
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/fundamentals-annual/

5. **Annual Remuneration - BoardEx (Wharton Data Research Services):**
    
    Data such as salary, bonus, and other cash compensation
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/compensation-analysis/annual-remuneration/

6. **Firms in Stoxx 600 Index**

    STOXX 600 is a major stock index representing the performance of 600 large-, mid-, and small-capitalization companies across 17 developed European countries.

    https://www.stoxx.com/selection-lists

**Table grains:** Employment data are role-spell records, remuneration data are executive-year records, fundamentals are firm-year records, and market prices are firm-security-month records. The final analysis table is reduced to one selected CEO observation per firm-year.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Total CEO compensation | Numeric | **Target** | Total annual compensation retained in reported currency, nominal EUR, and real 2015 EUR |
| Pay dispersion | Numeric | **Target / Outcome** | CEO pay relative to top-executive team (e.g., CEO-to-top-5 ratio) |
| CEO turnover indicator | Binary | **Treatment** | Flag for CEO change in a given year (from BoardEx role start/end) |
| Post‑turnover period | Binary | Feature | Indicator for years after turnover (event window) |
| Firm size (log assets / market cap) | Numeric | Feature | Scale and visibility of firm |
| Profitability (ROA / EBIT margin) | Numeric | Feature | Performance controls |
| Leverage | Numeric | Feature | Capital structure |
| Industry & country fixed effects | Categorical | Feature | Sector and institutional context |

**Planned outcomes and controls:**

The preparation pipeline constructs CEO compensation and turnover variables, converts monetary levels to nominal EUR, and deflates them to country-specific real 2015 EUR. Pay dispersion remains a planned extension before the final modelling stage.

**Potential data quality issues:**  
- **Missing compensation components:** use multiple imputation or restrict to firms with complete pay breakdowns; report sensitivity to this choice.
- **Turnover date ambiguity:** define CEO change using role start/end dates and validate with overlapping roles; conduct robustness with alternative windows.
- **Reporting bias / top-coding:** winsorize extreme pay values; compare distributions by country to detect systematic reporting differences.
- **Selection bias in BoardEx coverage:** include a Stoxx 600 filter and check representativeness vs. population benchmarks.
- **Timing misalignment:** align fiscal-year fundamentals with compensation year; drop or lag inconsistent observations.
- **Currency and inflation effects:** implemented through date-aligned EUR exchange rates followed by headquarters-country CPI rebased to 2015 = 100; observations outside CPI coverage remain missing.

---

<a id="helper-functions"></a>
## 4. Setup and Helper Functions

This section defines reusable display, timing, connection, and caching utilities. Keeping these operations in one place makes the data pipeline easier to rerun and audit.


In [ ]:
# Measure the running time of code blocks and print it in a readable format.

import time

def print_running_time(start_time, label="Running time"):
    """
    Print elapsed time since start_time.

    Parameters
    ----------
    start_time : float
        Start time from time.time()
    label : str
        Text label for the printed output
    """
    elapsed = time.time() - start_time

    if elapsed < 60:
        print(f"{label}: {elapsed:.2f} seconds")
    else:
        print(f"{label}: {elapsed:.2f} seconds ({elapsed / 60:.2f} minutes)")

In [ ]:
import pandas as pd

# Summarizes dataset health by calculating missing values per column and total duplicate rows.
def missing_and_duplicate_summary(df, sort=True):
    """
    Show missing values breakdown per column and high-level 
    dataset text metrics including duplicate row counts.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to inspect.
    sort : bool
        If True, sort columns by missing percentage descending.

    Returns
    -------
    pandas.DataFrame
        Table with missing counts and percentages.
    """
    # 1. Calculate general overview metrics
    total_rows = len(df)
    total_cells = df.size
    total_missing = df.isna().sum().sum()
    overall_completeness = ((total_cells - total_missing) / total_cells) * 100 if total_cells > 0 else 0
    
    # Calculate duplicate rows
    total_duplicates = df.duplicated().sum()
    duplicate_percent = (total_duplicates / total_rows) * 100 if total_rows > 0 else 0

    # 2. Print the high-level summary text
    print("==================================================")
    print("               DATA QUALITY SUMMARY               ")
    print("==================================================")
    print(f"Dataset shape       : {df.shape[0]:,} rows, {df.shape[1]:,} columns")
    print(f"Total data cells    : {total_cells:,}")
    print(f"Total missing values: {total_missing:,} ({overall_completeness:.2f}% populated)")
    print(f"Total duplicate rows: {total_duplicates:,} ({duplicate_percent:.2f}% of rows)")
    print("--------------------------------------------------\n")

    # 3. Calculate column-specific missing data
    missing_count = df.isna().sum()
    missing_percent = (missing_count / total_rows) * 100 if total_rows > 0 else 0

    missing_df = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    })

    missing_df["missing_percent"] = missing_df["missing_percent"].round(2)

    # 4. Filter, sort, and return
    if sort:
        missing_df = missing_df.sort_values(
            by="missing_percent",
            ascending=False
        )

    print("Detailed missing data breakdown per column:")
    return missing_df

In [ ]:
# show all columns and rows in DataFrame outputs
pd.set_option('display.max_columns', None)

In [ ]:
import pandas as pd
from pathlib import Path

def summarize_dataframe(df_input):
    """Display and return a dtype-safe health summary for a DataFrame or data file.

    Supports CSV, Excel, pickle, and Parquet files. Categorical statistics are
    calculated explicitly so pandas `string`, `object`, and `category` dtypes
    are handled consistently.
    """
    if isinstance(df_input, (str, Path)):
        path = Path(df_input)
        readers = {
            ".csv": pd.read_csv,
            ".xlsx": pd.read_excel,
            ".xls": pd.read_excel,
            ".pkl": pd.read_pickle,
            ".pickle": pd.read_pickle,
            ".parquet": pd.read_parquet,
        }
        if path.suffix.lower() not in readers:
            raise ValueError(
                f"Unsupported file extension: {path.suffix or '<none>'}. "
                f"Supported extensions: {sorted(readers)}"
            )
        df = readers[path.suffix.lower()](path)
        print(f"Loaded: {path}")
    elif isinstance(df_input, pd.DataFrame):
        df = df_input
    else:
        raise TypeError("df_input must be a pandas DataFrame or a supported file path.")

    if df.empty:
        print(f"Empty DataFrame with {df.shape[1]} columns.")
        return {"overview": pd.DataFrame(), "missing": pd.DataFrame()}

    try:
        duplicate_rows = int(df.duplicated().sum())
    except TypeError:
        duplicate_rows = pd.NA

    overview = pd.DataFrame({
        "value": [
            len(df),
            df.shape[1],
            duplicate_rows,
            f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
        ]
    }, index=["rows", "columns", "duplicate rows", "memory usage"])

    missing = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()
    datetime_cols = df.select_dtypes(
        include=["datetime", "datetimetz"]
    ).columns.tolist()

    numeric_summary = (
        df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()
    )

    categorical_rows = []
    for col in categorical_cols:
        counts = df[col].value_counts(dropna=True)
        categorical_rows.append({
            "column": col,
            "count": int(df[col].notna().sum()),
            "missing": int(df[col].isna().sum()),
            "unique": int(df[col].nunique(dropna=True)),
            "top": counts.index[0] if not counts.empty else pd.NA,
            "top_frequency": int(counts.iloc[0]) if not counts.empty else 0,
        })
    categorical_summary = (
        pd.DataFrame(categorical_rows).set_index("column")
        if categorical_rows else pd.DataFrame()
    )

    datetime_summary = pd.DataFrame()
    if datetime_cols:
        datetime_summary = pd.DataFrame({
            "min": df[datetime_cols].min(),
            "max": df[datetime_cols].max(),
            "missing": df[datetime_cols].isna().sum(),
        })

    print("### Dataset overview")
    display(overview)
    print("### Column coverage and dtypes")
    display(missing)
    if not numeric_summary.empty:
        print("### Numeric columns")
        display(numeric_summary)
    if not categorical_summary.empty:
        print("### Categorical columns")
        display(categorical_summary)
    if not datetime_summary.empty:
        print("### Datetime columns")
        display(datetime_summary)

    return {
        "overview": overview,
        "missing": missing,
        "numeric": numeric_summary,
        "categorical": categorical_summary,
        "datetime": datetime_summary,
    }


<a id="data-collection"></a>
## 5. Data Collection and Profiling

The data pipeline follows a common STOXX 600 sample frame:

1. STOXX 600 ISINs define the firm universe.
2. BoardEx employment records provide CEO roles and turnover dates.
3. BoardEx executive details provide demographic characteristics.
4. BoardEx remuneration records provide annual compensation.
5. Compustat annual fundamentals provide firm-year controls.
6. Compustat monthly security data provide prices for market capitalization.
7. Compustat daily exchange rates provide the currency bridge to EUR.

Each query selects only columns used by the preparation or modelling pipeline. This reduces WRDS transfer time, cache size, and in-memory DataFrame size while keeping identifiers needed for validation. The sources are combined only after their identifiers, dates, currencies, and table grains have been checked.


<a id="stoxx-600-universe"></a>
### 5.1 STOXX 600 Company Universe


Before querying BoardEx, we define the project sample. The analysis is restricted to firms in the STOXX Europe 600. We use the `stoxx600_clean.csv` file as the sample frame and extract its ISINs as the main identifier for WRDS queries.

In [ ]:
# Core packages and project paths
import wrds
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/data_project")
STOXX_FILE = DATA_DIR / "stoxx600_clean.csv"
WRDS_CACHE_DIR = DATA_DIR / "wrds_cache"
WRDS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set this to True when you deliberately want to replace all cached WRDS data.
REFRESH_WRDS_CACHE = False

def get_wrds_connection():
    """Create the WRDS connection only when a server query is actually needed."""
    global conn

    try:
        conn
        print("Using existing WRDS connection.")
    except NameError:
        conn = wrds.Connection()
        print("Created new WRDS connection.")

    return conn


def load_or_fetch(cache_name, fetch_function, refresh=False):
    """Load a DataFrame from disk, or fetch and cache it when unavailable.

    Parameters
    ----------
    cache_name : str
        File stem used inside WRDS_CACHE_DIR.
    fetch_function : callable
        Zero-argument function that retrieves and returns a DataFrame.
    refresh : bool
        If True, ignore any existing cache and retrieve fresh data.
    """
    cache_path = WRDS_CACHE_DIR / f"{cache_name}.pkl"

    if cache_path.exists() and not refresh:
        try:
            df = pd.read_pickle(cache_path)
            print(f"Loaded cached data: {cache_path} ({len(df):,} rows)")
            return df
        except (AttributeError, ImportError, ModuleNotFoundError, TypeError, ValueError, EOFError) as exc:
            print(
                f"Cache could not be read ({type(exc).__name__}); "
                "retrieving a compatible copy from WRDS."
            )

    print(f"Cache not used; retrieving data from WRDS: {cache_name}")
    df = fetch_function()

    if not isinstance(df, pd.DataFrame):
        raise TypeError("fetch_function must return a pandas DataFrame.")

    # Write to a temporary file first so an interrupted save cannot corrupt the cache.
    temporary_path = cache_path.with_suffix(".tmp")
    df.to_pickle(temporary_path)
    temporary_path.replace(cache_path)
    print(f"Saved data to cache: {cache_path} ({len(df):,} rows)")

    return df

print(f"STOXX 600 file exists: {STOXX_FILE.exists()}")
print(f"WRDS cache directory: {WRDS_CACHE_DIR.resolve()}")

WRDS results are cached as pickle files in `../data/wrds_cache`. On later runs, the notebook loads those local files and does not open a WRDS connection. A connection is created lazily only when a cache is missing or `REFRESH_WRDS_CACHE = True`.

In [ ]:
# No connection is opened here. Each retrieval cell first checks its local cache.
print(f"Cached WRDS datasets available: {len(list(WRDS_CACHE_DIR.glob('*.pkl')))}")
print("A WRDS connection will be opened only if a required cache is unavailable.")

Only `ISIN` is required from the STOXX 600 file: it defines the firm universe and serves as the filter for BoardEx and Compustat. The values are standardized before they are passed to WRDS.

In [ ]:
# Read STOXX Europe 600 constituents
df_sxxp = pd.read_csv(STOXX_FILE, sep=";", usecols=["ISIN"])

df_sxxp["ISIN"] = (
    df_sxxp["ISIN"]
    .astype(str)
    .str.strip()
    .str.upper()
)

isin_list = df_sxxp["ISIN"].dropna().unique().tolist()

print(f"STOXX 600 rows: {len(df_sxxp):,}")
print(f"Unique STOXX 600 ISINs: {len(isin_list):,}")
print(f"Duplicate ISIN rows: {df_sxxp['ISIN'].duplicated().sum():,}")

df_sxxp.head()

| Column | Precise description | WRDS/pandas representation |
|---|---|---|
| `ISIN` | Twelve-character International Securities Identification Number used to restrict all server queries to the STOXX Europe 600 universe. | String |

In [ ]:
summarize_dataframe(df_sxxp)

**Table description:** `df_sxxp` is a one-column sample-frame table. Its grain is one STOXX constituent per ISIN. The summary verifies identifier coverage and confirms that duplicate ISINs will not create repeated query keys.


These ISINs are the bridge from the index universe to BoardEx. All following BoardEx employment and remuneration pulls should be filtered using this STOXX 600 firm list, so the project remains focused on European large- and mid-cap firms.

<a id="boardex-employment-records"></a>
### 5.2 BoardEx Employment Records


The STOXX 600 file defines the company universe for the project. We use the ISINs from `df_sxxp` as the filter for BoardEx Europe employment records, so all later CEO turnover and compensation variables are built only for firms that are members of the STOXX 600 sample.

This query retains only the person, company, role-spell, and firm-context fields used downstream. Role start and end dates define CEO spells; `directorid` identifies CEO changes; `companyid` and `isin` provide merge keys; and sector/country fields become final-panel controls.

In [ ]:
start_time = time.time()

# Pull BoardEx employment records for the STOXX 600 firm universe when not cached
isin_list = df_sxxp["ISIN"].dropna().str.strip().str.upper().unique().tolist()

def fetch_boardex_employment():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(isin_list), chunk_size):
        isin_chunk = tuple(isin_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT
                companyid, companyname, directorid, directorname,
                datestartrole, dateendrole, rolename,
                hocountryname, sector, isin
            FROM boardex.eur_wrds_dir_profile_emp
            WHERE isin IN %(isins)s
            ORDER BY companyname, directorname, datestartrole, rolename
        """, params={"isins": isin_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_emp = load_or_fetch(
    "boardex_employment_stoxx600_analysis_cols_v1",
    fetch_boardex_employment,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 ISINs used for WRDS query: {len(isin_list):,}")
print(f"BoardEx employment records retrieved: {len(df_emp):,}")
print(f"Unique companies matched in BoardEx: {df_emp['companyid'].nunique():,}")
print(f"Unique individuals matched in BoardEx: {df_emp['directorid'].nunique():,}")

print_running_time(start_time, label="Employment data load time")

df_emp.head()

**Table description:** `df_emp` has one row per BoardEx employment-role spell. It contains the minimum fields required to identify firm-level CEOs, order their spells through time, construct turnover events, link people and firms to other sources, and attach sector and headquarters-country controls.


| Column | Precise description | WRDS type |
|---|---|---|
| `companyid` | BoardEx company identifier used to group CEO spells and merge remuneration records (`boardid`). | Float |
| `companyname` | BoardEx company name retained for readable diagnostics and outputs. | Char(128) |
| `directorid` | Stable BoardEx individual identifier used to identify consecutive CEOs and merge executive/remuneration records. | Float |
| `directorname` | Individual name retained for readable CEO-spell and transition checks. | Char(255) |
| `datestartrole` | First date of the employment-role spell; determines CEO ordering and turnover year. | Date |
| `dateendrole` | Last date of the role spell; open-ended sentinel dates are cleaned later. | Date |
| `rolename` | Reported role title used to identify and refine firm-level CEO positions. | Char(100) |
| `hocountryname` | Company headquarters country used as a country control in the final panel. | Char(255) |
| `sector` | BoardEx company-sector classification used as an industry control. | Char(100) |
| `isin` | International Securities Identification Number linking BoardEx firms to the STOXX and Compustat samples. | Char(12) |

In [ ]:
summarize_dataframe(df_emp)

**Insights:**

- BoardEx matches slightly fewer firms than the 600 STOXX constituents, which is expected because coverage depends on BoardEx identifiers and available employment histories.
- The employment table is the backbone of the turnover design: it supplies `directorid`, `companyid`, role titles, and start/end dates.
- The most important quality check here is whether CEO-like role titles are too broad; later filters narrow regional or divisional CEO roles into firm-level CEO spells.


<a id="boardex-executive-details"></a>
### 5.3 BoardEx Executive Details

The employment table identifies which people belong to the sample. Only `directorid`, age, and gender are retrieved because these are the executive-profile fields used in the final analysis panel.

| Column | Precise description | WRDS type |
|---|---|---|
| `directorid` | BoardEx individual identifier used to merge the profile onto CEO records. | Float |
| `age` | Director age reported by BoardEx at the time of data extraction. It is treated as an executive characteristic, not a historical age-by-year measure. | Char(3) |
| `gender` | One-character gender classification reported by BoardEx. | Char(1) |
|`dob` | Date of birth | Date |
|`dod` | Date of death | Date |


In [ ]:
start_time = time.time()

# Retrieve executive details for all matched directors when not cached
directorid_list = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

def fetch_boardex_executive_details():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(directorid_list), chunk_size):
        directorid_chunk = tuple(directorid_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT directorid, age, gender, dob, dod
            FROM boardex.eur_dir_profile_details
            WHERE directorid IN %(directorids)s
            ORDER BY directorname
        """, params={"directorids": directorid_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_exec = load_or_fetch(
    "boardex_executive_details_stoxx600_analysis_cols_v1",
    fetch_boardex_executive_details,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"Director IDs queried: {len(directorid_list):,}")
print(f"Executive records retrieved: {len(df_exec):,}")
print(f"Unique executives: {df_exec['directorid'].nunique():,}")

print_running_time(start_time, label="Executive details load time")

df_exec

In [ ]:
summarize_dataframe(df_exec)

<a id="boardex-remuneration-records"></a>
### 5.4 BoardEx Remuneration Records

We retrieve remuneration for all matched executives rather than CEOs alone. The selected fields cover identifiers, reporting date and currency, core cash compensation, and the equity/performance measures retained in the CEO-pay panel. The broader executive population remains available for the planned pay-dispersion outcome.

| Column | Precise description | WRDS type / unit |
|---|---|---|
| `boardid` | BoardEx company identifier; matched to employment `companyid`. | Float |
| `boardname` | Company/board name used for readable matching diagnostics. | Char(128) |
| `directorid` | BoardEx individual identifier linking remuneration to CEO spells. | Float |
| `directorname` | Individual name used for readable matching diagnostics. | Char(255) |
| `rolename` | Role reported on the remuneration record; retained to distinguish it from the employment-role title after merging. | Char(100) |
| `annualreportdate` | Annual-report date associated with the remuneration observation; its year defines the pay-panel year. | Date |
| `currency` | Three-character currency code supplied by BoardEx for the monetary fields. | Char(3) |
| `salary` | Annual base salary reported by BoardEx. | Float; USD thousands in the current extract |
| `bonus` | Annual cash bonus reported by BoardEx. | Float; USD thousands in the current extract |
| `totalcompensation` | Main CEO-pay outcome: total annual compensation reported by BoardEx. | Float; thousands |
| `totaldirectcomp` | Salary + bonus + other compensation + employer pension contribution. | Float; thousands |
| `perftotal` | LTIP value divided by total awards for the remuneration period. | Float ratio |
| `valtoteqheld` | Total value of equity held by the executive. | Float; thousands |
| `valltipheld` | Value of long-term incentive plans held. | Float; thousands |
| `valeqaward` | Value of equity awarded during the latest year. | Float; thousands |
| `ltipvalue` | Value of LTIP awards granted during the latest year. | Float; thousands |
| `toteqatrisk` | Total value of stock, option, and LTIP awards at risk. | Float; thousands |


In [ ]:
start_time = time.time()

# Retrieve remuneration for all executives when not cached (needed for pay dispersion)
directorid_list = tuple(df_emp["directorid"].dropna().astype(int).unique().tolist())

def fetch_boardex_remuneration():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(directorid_list), chunk_size):
        directorid_chunk = tuple(directorid_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT
                boardname, boardid, directorname, directorid, rolename,
                annualreportdate, currency, salary, bonus, totalcompensation,
                valtoteqheld, valltipheld, valeqaward, ltipvalue, toteqatrisk,
                totaldirectcomp, perftotal
            FROM boardex.eur_dir_standard_remun
            WHERE directorid IN %(directorids)s
            ORDER BY boardname, directorname, annualreportdate
        """, params={"directorids": directorid_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_remun = load_or_fetch(
    "boardex_remuneration_stoxx600_analysis_cols_v1",
    fetch_boardex_remuneration,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"Director IDs queried: {len(directorid_list):,}")
print(f"Remuneration records retrieved: {len(df_remun):,}")
print(f"Unique remunerated directors: {df_remun['directorid'].nunique():,}")

print_running_time(start_time, label="Remuneration data load time")

df_remun.head()

**Table description:** `df_remun` has one row per executive, board, and annual-report observation where BoardEx provides a remuneration record. Coverage is sparse by design, so the profile emphasizes missingness and currency consistency before CEO matching. Monetary fields retain BoardEx's reported scale; conversion or inflation adjustment must therefore preserve their documented units.


In [ ]:
summarize_dataframe(df_remun)

<a id="compustat-fundamentals"></a>
### 5.5 Compustat Annual Fundamentals

Compustat fundamentals add the annual firm controls used in `df_panel_final`. The compact query retains identifiers and dates, accounting currency, the numerators and denominators required for profitability and leverage ratios, employment, and shares outstanding for market capitalization.


In [ ]:
start_time = time.time()

# Pull annual firm fundamentals for the STOXX 600 universe when not cached
isin_list = df_sxxp["ISIN"].dropna().str.strip().str.upper().unique().tolist()

def fetch_compustat_fundamentals():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(isin_list), chunk_size):
        isin_chunk = tuple(isin_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT
                gvkey, isin, fyear, datadate, curcd,
                at, sale, ebit, dltt, nicon, emp, cshoi
            FROM comp_global_daily.g_funda
            WHERE isin IN %(isins)s
              AND fyear BETWEEN 2000 AND 2024
            ORDER BY isin, fyear, datadate
        """, params={"isins": isin_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_firm = load_or_fetch(
    "compustat_fundamentals_stoxx600_2000_2024_analysis_cols_v1",
    fetch_compustat_fundamentals,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 ISINs used for WRDS query: {len(isin_list):,}")
print(f"Firm fundamentals records retrieved: {len(df_firm):,}")
print(f"Unique ISINs matched: {df_firm['isin'].nunique():,}")

print_running_time(start_time, label="Fundamentals data load time")

df_firm.head()

**Table description:** `df_firm` has one annual accounting observation per Compustat firm/security identifier and fiscal year. The selected variables are exactly those needed to merge firms, align reporting dates, construct ROA, EBIT margin, leverage and log assets, control for employment, and calculate market capitalization after attaching prices.


| Column | Precise description | Compustat representation |
|---|---|---|
| `gvkey` | Stable Compustat company identifier used to connect annual fundamentals to monthly security prices. | Char |
| `isin` | Security identifier used to connect Compustat fundamentals to the STOXX/BoardEx firm sample. | Char |
| `fyear` | Fiscal year assigned by Compustat; renamed `annualreportyear` in the final merge. | Integer |
| `datadate` | Fiscal reporting date used to attach the nearest monthly market price within 31 days. | Date |
| `curcd` | Currency in which the annual accounting values are reported. | Char |
| `at` | Total assets; denominator for ROA and leverage and input to log assets. | Numeric |
| `sale` | Net sales/turnover; denominator for EBIT margin. | Numeric |
| `ebit` | Earnings before interest and taxes; numerator for EBIT margin. | Numeric |
| `dltt` | Long-term debt; numerator for the leverage ratio. | Numeric |
| `nicon` | Consolidated net income/loss; numerator for ROA. | Numeric |
| `emp` | Number of employees, generally reported in thousands by Compustat. | Numeric |
| `cshoi` | Common shares outstanding for the issue, used with monthly price to calculate market capitalization. | Numeric |

<a id="compustat-market-prices"></a>
### 5.6 Compustat Monthly Market Prices

Monthly security prices are retrieved for the `gvkey` values present in annual fundamentals. The compact table keeps only the firm and issue identifiers, observation date, closing price, price currency, and exchange code needed to select one listing and construct market capitalization. Its grain is one security issue per month.


In [ ]:
start_time = time.time()

# Pull monthly security prices for the matched firms when not cached
gvkey_list = (
    df_firm["gvkey"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
    .tolist()
)

def fetch_compustat_market_data():
    wrds_conn = get_wrds_connection()
    chunks = []
    chunk_size = 1000

    for i in range(0, len(gvkey_list), chunk_size):
        gvkey_chunk = tuple(gvkey_list[i:i + chunk_size])
        temp = wrds_conn.raw_sql("""
            SELECT gvkey, iid, datadate, prccm, curcdm, exchg
            FROM comp.g_secm
            WHERE gvkey IN %(gvkeys)s
            ORDER BY gvkey, datadate
        """, params={"gvkeys": gvkey_chunk})
        chunks.append(temp)

    return pd.concat(chunks, ignore_index=True)


df_market = load_or_fetch(
    "compustat_monthly_market_stoxx600_analysis_cols_v1",
    fetch_compustat_market_data,
    refresh=REFRESH_WRDS_CACHE,
)

print(f"STOXX 600 GVKEYs used for WRDS query: {len(gvkey_list):,}")
print(f"Monthly market records available: {len(df_market):,}")
print(f"Unique GVKEYs matched: {df_market['gvkey'].nunique():,}")

print_running_time(start_time, label="Market data load time")

df_market.head()

| Column | Precise description | Compustat representation |
|---|---|---|
| `gvkey` | Compustat company identifier linking monthly prices to annual fundamentals. | Char |
| `iid` | Compustat issue identifier distinguishing multiple securities issued by the same company. | Char |
| `datadate` | Month-end date of the security-price observation. | Date |
| `prccm` | Monthly closing price in the currency identified by `curcdm`. | Numeric |
| `curcdm` | ISO currency code applicable to `prccm`. | Char |
| `exchg` | Compustat exchange code used to select one primary price history per `gvkey`. | Integer |

In [ ]:
summarize_dataframe(df_market)

<a id="compustat-exchange-rates"></a>
### 5.7 Compustat Daily Exchange Rates

The analysis requires a common currency for monetary levels. Source currencies are derived directly from remuneration, annual fundamentals, and monthly prices. Compustat stores these observations as GBP-based cross rates, so the query retrieves only GBP-to-required-currency series and the function derives each source-currency-to-EUR rate as `(GBP→EUR) / (GBP→source currency)`. This avoids loading the full cross-currency matrix. The resulting table grain is one source-currency/date conversion rate to EUR.


In [ ]:
# Retrieve only currencies observed in the monetary source tables.
currency_list = sorted(
    set(df_remun["currency"].dropna().astype(str))
    | set(df_firm["curcd"].dropna().astype(str))
    | set(df_market["curcdm"].dropna().astype(str))
)

def fetch_comp_gcurrency():
    """Fetch GBP cross-rates and derive units of EUR per source currency."""
    wrds_conn = get_wrds_connection()
    target_currencies = tuple(sorted(set(currency_list) | {"EUR", "GBP"}))

    raw_cross_rates = wrds_conn.raw_sql(
        """
        SELECT
            datadate,
            fromcurd,
            tocurd,
            exratd
        FROM comp.g_exrt_dly
        WHERE fromcurd = 'GBP'
          AND tocurd IN %(currencies)s
          AND datadate BETWEEN '1995-01-01' AND '2025-12-31'
        ORDER BY datadate, tocurd
        """,
        params={"currencies": target_currencies},
    )

    raw_cross_rates["datadate"] = pd.to_datetime(
        raw_cross_rates["datadate"], errors="coerce"
    )
    rate_matrix = raw_cross_rates.pivot_table(
        index="datadate",
        columns="tocurd",
        values="exratd",
        aggfunc="last",
    )

    if "EUR" not in rate_matrix.columns:
        raise ValueError("Compustat returned no GBP-to-EUR reference series.")

    # If 1 GBP = x EUR and 1 GBP = y source-currency units,
    # then 1 source-currency unit = x / y EUR.
    eur_rates = rate_matrix.rdiv(rate_matrix["EUR"], axis=0)
    eur_rates.columns.name = "fromcurd"

    result = (
        eur_rates
        .stack(future_stack=True)
        .rename("exratd")
        .reset_index()
    )
    result["tocurd"] = "EUR"
    return result[["datadate", "fromcurd", "tocurd", "exratd"]]


df_gcurrency = load_or_fetch(
    "compustat_daily_fx_to_eur_analysis_currencies_1995_2025_v2",
    fetch_comp_gcurrency,
    refresh=REFRESH_WRDS_CACHE,
)

df_gcurrency["datadate"] = pd.to_datetime(
    df_gcurrency["datadate"], errors="coerce"
)

# Average rates are appropriate for annual flow variables such as pay, sales, and earnings.
fx_annual_average = (
    df_gcurrency
    .assign(fx_year=df_gcurrency["datadate"].dt.year)
    .groupby(["fromcurd", "fx_year"], as_index=False)["exratd"]
    .mean()
    .rename(columns={"exratd": "fx_rate_avg_to_eur"})
)


def attach_daily_eur_rate(df, date_col, currency_col, rate_col, tolerance_days=7):
    """Attach the latest available EUR rate on or before each observation date."""
    left = df.copy()
    left[date_col] = (
        pd.to_datetime(left[date_col], errors="coerce")
        .astype("datetime64[ns]")
    )
    left[currency_col] = left[currency_col].astype("string")

    right = (
        df_gcurrency[["datadate", "fromcurd", "exratd"]]
        .rename(columns={
            "datadate": "fx_date",
            "fromcurd": currency_col,
            "exratd": rate_col,
        })
        .dropna(subset=["fx_date", currency_col, rate_col])
    )
    right["fx_date"] = pd.to_datetime(right["fx_date"], errors="coerce").astype("datetime64[ns]")
    right[currency_col] = right[currency_col].astype("string")

    return pd.merge_asof(
        left.sort_values([date_col, currency_col]),
        right.sort_values(["fx_date", currency_col]),
        left_on=date_col,
        right_on="fx_date",
        by=currency_col,
        direction="backward",
        tolerance=pd.Timedelta(days=tolerance_days),
    )

df_gcurrency.head()

| Column | Precise description | Compustat representation |
|---|---|---|
| `datadate` | Calendar date to which the daily exchange rate applies. | Date |
| `fromcurd` | ISO code of the monetary variable's original currency. | Char |
| `tocurd` | Target currency; fixed to EUR by this query. | Char |
| `exratd` | Units of EUR obtained for one unit of `fromcurd`; multiply the original amount by this rate to express it in EUR. | Numeric |

In [ ]:
summarize_dataframe(df_gcurrency)

---

<a id="ceo-turnover-and-pay-panel"></a>
## 6. CEO Turnover and Pay Panel

This section converts raw role and remuneration records into a firm-year CEO panel. The sequence is deliberate: identify valid CEO spells, detect changes between consecutive CEOs, match pay to the correct person and firm, and finally construct treatment and outcome variables.


<a id="ceo-turnover-events"></a>
### 6.1 Define CEO Turnover Events


#### 6.1.1 Identify broad CEO candidates from role names

In [ ]:
# Identify broad CEO candidates from role names
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|chief executive officer|managing director"

# Clean dates before constructing CEO spells and turnover events
df_emp = df_emp.copy()
df_emp["datestartrole"] = pd.to_datetime(df_emp["datestartrole"], errors="coerce")

df_emp["dateendrole_clean"] = (
    df_emp["dateendrole"]
    .astype(str)
    .replace(["9000-01-01", "9999-12-31", "NaT", "nan", "None"], pd.NA)
)

df_emp["dateendrole_clean"] = pd.to_datetime(
    df_emp["dateendrole_clean"],
    errors="coerce"
)

df_emp["rolename_clean"] = df_emp["rolename"].fillna("").str.strip()

df_emp["is_ceo"] = (
    df_emp["rolename_clean"]
    .str.lower()
    .str.contains(CEO_PATTERN, na=False, regex=True)
)

df_emp_ceo = df_emp[df_emp["is_ceo"]].copy()

print(f"Broad CEO candidate role records: {len(df_emp_ceo):,}")
df_emp_ceo["rolename_clean"].value_counts().head(30)

#### 6.1.2 Refine candidates to firm-level CEO spells

In [ ]:
# The broad CEO keyword search also captures regional, divisional, acting, and advisory roles.
df_emp_ceo = df_emp_ceo.copy()

# Main firm-level CEO keywords
has_main_ceo_title = df_emp_ceo["rolename_clean"].str.contains(
    r"\bCEO\b|Chief Executive|Chief Executive Officer|Group CEO|President/CEO|Chairman/CEO|Chair/CEO|MD/CEO|CEO/MD|Managing Director",
    case=False,
    regex=True,
    na=False
)

# Exclude business-unit / regional / lower-level CEO roles
exclude_unit_roles = df_emp_ceo["rolename_clean"].str.contains(
    r"Regional|Division|Divisional|Country|Branch|Sector|Zonal|Area|Business Unit|Market|Subsidiary|Segment",
    case=False,
    regex=True,
    na=False
)

# Exclude non-actual CEO roles
exclude_non_actual = df_emp_ceo["rolename_clean"].str.contains(
    r"Designate|Elect|in-Residence|Delegate|Office|Adviser|Advisor|Honorary|Shareholder Representative",
    case=False,
    regex=True,
    na=False
)

# Exclude subordinate CEO roles
exclude_subordinate = df_emp_ceo["rolename_clean"].str.contains(
    r"Deputy CEO|Vice CEO|Associate CEO|Assistant CEO",
    case=False,
    regex=True,
    na=False
)

# Interim / acting CEOs are excluded from the strict baseline definition
is_interim_acting = df_emp_ceo["rolename_clean"].str.contains(
    r"Interim|Acting",
    case=False,
    regex=True,
    na=False
)

# Co-CEO / joint CEO can be included in a robustness definition
is_co_ceo = df_emp_ceo["rolename_clean"].str.contains(
    r"Co-CEO|Joint CEO",
    case=False,
    regex=True,
    na=False
)

# Baseline CEO role definition
df_emp_ceo["firm_level_ceo_role"] = (
    has_main_ceo_title
    & ~exclude_unit_roles
    & ~exclude_non_actual
    & ~exclude_subordinate
)

# Strict baseline: exclude interim/acting and co-CEO roles
df_emp_ceo["firm_level_ceo_strict"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
    & ~is_co_ceo
)

# Robustness definition: include co-CEOs but exclude interim/acting CEOs
df_emp_ceo["firm_level_ceo_with_coceo"] = (
    df_emp_ceo["firm_level_ceo_role"]
    & ~is_interim_acting
)

print(f"Strict firm-level CEO spells: {df_emp_ceo['firm_level_ceo_strict'].sum():,}")
print(f"Firm-level CEO spells including co-CEOs: {df_emp_ceo['firm_level_ceo_with_coceo'].sum():,}")

df_emp_ceo

#### 6.1.3 Construct the baseline CEO-spell table

In [ ]:
df_ceo_main = (
    df_emp_ceo[df_emp_ceo["firm_level_ceo_strict"]]
    .dropna(subset=["companyid", "directorid", "datestartrole"])
    .sort_values(["companyid", "datestartrole", "dateendrole_clean", "directorid"])
    .copy()
)

df_ceo_main

#### 6.1.4 Identify CEO turnover events

CEO turnover is defined when the current firm-level CEO's `directorid` differs from the previous observed CEO's `directorid` within the same company. The first observed CEO spell per company is not coded as turnover, because we do not observe a prior CEO inside our sample window.

In [ ]:
# Using .ne(...).fillna(False) avoids pandas NA-to-integer conversion errors.
df_ceo_main["previous_ceo_directorid"] = (
    df_ceo_main
    .groupby("companyid")["directorid"]
    .shift()
)

df_ceo_main["ceo_turnover"] = (
    df_ceo_main["directorid"]
    .ne(df_ceo_main["previous_ceo_directorid"])
    .fillna(False)
    .astype(int)
)

# First observed CEO in each company is not counted as turnover
df_ceo_main.loc[
    df_ceo_main["previous_ceo_directorid"].isna(),
    "ceo_turnover"
] = 0

df_ceo_main["turnover_year"] = df_ceo_main["datestartrole"].dt.year

print(f"CEO role spells: {len(df_ceo_main):,}")
print(f"Unique companies with CEO spells: {df_ceo_main['companyid'].nunique():,}")
print(f"CEO turnover events: {df_ceo_main['ceo_turnover'].sum():,}")

df_ceo_main[[
    "companyname", "directorname", "rolename", "datestartrole",
    "dateendrole_clean", "previous_ceo_directorid", "ceo_turnover", "turnover_year"
]].head(10)

#### 6.1.5 Collapse turnover events to the firm-year level

In [ ]:
ceo_turnover_firm_year = (
    df_ceo_main[df_ceo_main["ceo_turnover"] == 1]
    .groupby(["companyid", "turnover_year"])
    .size()
    .reset_index(name="num_ceo_turnovers")
)

ceo_turnover_firm_year["ceo_turnover_dummy"] = 1
ceo_turnover_firm_year.head(20)

The resulting `ceo_turnover_firm_year` table will later be merged into the remuneration panel by `companyid` and fiscal/report year.

<a id="ceo-remuneration-matching"></a>
### 6.2 Prepare Annual Remuneration Records for Identified CEOs


The cached remuneration table contains all executives in the sampled firms. We now filter it to the `directorid` values identified as firm-level CEOs, clean the compensation variables, and then match each record to the correct CEO spell and company.

In [ ]:
df_remun

#### 6.2.1 Filter remuneration to identified CEOs

In [ ]:
ceo_directorid_list = (
    df_ceo_main["directorid"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

df_remun_ceos = df_remun[df_remun["directorid"].isin(ceo_directorid_list)].copy()

print(f"CEO director IDs queried: {len(ceo_directorid_list):,}")
print(f"Remuneration records retrieved: {len(df_remun_ceos):,}")
print(f"Unique remunerated directors: {df_remun_ceos['directorid'].nunique():,}")
print(f"Remuneration records for CEOs: {len(df_remun_ceos):,}")

df_remun_ceos

We first clean dates and monetary fields. Because remuneration represents compensation accumulated over a reporting year, nominal values are then converted to EUR using the average daily exchange rate for that calendar year. Original reported values and currency codes are preserved beside the EUR variables.

<a id="remuneration-currency-conversion"></a>
#### 6.2.2 Clean Remuneration and Convert Nominal Values to EUR

In [ ]:
df_remun_ceos = df_remun_ceos.copy()

df_remun_ceos["annualreportdate"] = pd.to_datetime(
    df_remun_ceos["annualreportdate"],
    errors="coerce"
)

df_remun_ceos["annualreportyear"] = df_remun_ceos["annualreportdate"].dt.year

pay_cols = [
    "salary", "bonus", "totalcompensation", "valtoteqheld", "valltipheld",
    "valeqaward", "ltipvalue", "toteqatrisk",
    "totaldirectcomp", "perftotal"
]

for col in pay_cols:
    if col in df_remun_ceos.columns:
        df_remun_ceos[col] = pd.to_numeric(df_remun_ceos[col], errors="coerce")

# Attach the annual-average rate: units of EUR per unit of reported currency.
df_remun_ceos = (
    df_remun_ceos
    .drop(columns=["fromcurd", "fx_year", "remun_fx_rate_avg_to_eur"], errors="ignore")
    .merge(
        fx_annual_average.rename(columns={
            "fx_rate_avg_to_eur": "remun_fx_rate_avg_to_eur"
        }),
        left_on=["currency", "annualreportyear"],
        right_on=["fromcurd", "fx_year"],
        how="left",
        validate="many_to_one",
    )
    .drop(columns=["fromcurd", "fx_year"])
)

pay_monetary_cols = [col for col in pay_cols if col != "perftotal"]
for col in pay_monetary_cols:
    df_remun_ceos[f"{col}_eur"] = (
        df_remun_ceos[col] * df_remun_ceos["remun_fx_rate_avg_to_eur"]
    )

print("Missing annual report dates:", df_remun_ceos["annualreportdate"].isna().sum())
print("Missing total compensation:", df_remun_ceos["totalcompensation"].isna().sum())
print("Positive total compensation records:", (df_remun_ceos["totalcompensation"] > 0).sum())
usable_pay_for_fx = (
    df_remun_ceos["annualreportdate"].notna()
    & df_remun_ceos["totalcompensation"].gt(0)
)
print(
    "Missing FX rates among usable positive-pay records:",
    df_remun_ceos.loc[usable_pay_for_fx, "remun_fx_rate_avg_to_eur"].isna().sum(),
)

df_remun_ceos[[
    "directorid", "directorname", "boardid", "boardname",
    "annualreportdate", "currency", "remun_fx_rate_avg_to_eur",
    "totalcompensation", "totalcompensation_eur",
]].head(10)

### 6.3 Match CEO Remuneration to Valid CEO Spells


The merge first links remuneration and employment by `directorid`, then requires `boardid == companyid` so compensation cannot be assigned to the same person at another firm. We retain annual-report dates inside the CEO spell with a 90-day boundary tolerance. The tolerance accommodates reporting dates close to a transition, but it can allow both outgoing and incoming CEOs to appear in the same firm-year; the next section therefore applies an explicit one-record selection rule.

In [ ]:
ceo_spells = df_ceo_main[[
    "companyid", "companyname", "directorid", "directorname", "rolename",
    "datestartrole", "dateendrole_clean", "ceo_turnover", "turnover_year"
]].copy()

# Ensure merge keys are comparable
ceo_spells["companyid"] = pd.to_numeric(ceo_spells["companyid"], errors="coerce")
df_remun_ceos["boardid"] = pd.to_numeric(df_remun_ceos["boardid"], errors="coerce")


# df_rt_ceo = remuneration records for CEO candidates matched to valid CEO spells
df_rt_ceos = df_remun_ceos.merge(
    ceo_spells,
    on="directorid",
    how="inner",
    suffixes=("_remun", "_emp")
)

# Keep compensation records for the same firm only
df_rt_ceos = df_rt_ceos[
    df_rt_ceos["boardid"].eq(df_rt_ceos["companyid"])
].copy()

# Keep compensation records during the valid CEO role period
tolerance_days = 90
start_ok = df_rt_ceos["annualreportdate"] >= (
    df_rt_ceos["datestartrole"] - pd.Timedelta(days=tolerance_days)
)
end_ok = (
    df_rt_ceos["dateendrole_clean"].isna()
    | (df_rt_ceos["annualreportdate"] <= df_rt_ceos["dateendrole_clean"] + pd.Timedelta(days=tolerance_days))
)

df_rt_ceos = df_rt_ceos[start_ok & end_ok].copy()

# Keep usable compensation observations for the baseline CEO pay analysis
df_rt_ceos = df_rt_ceos[
    df_rt_ceos["annualreportdate"].notna()
    & df_rt_ceos["totalcompensation_eur"].notna()
    & (df_rt_ceos["totalcompensation_eur"] > 0)
].drop_duplicates().copy()

df_rt_ceos["annualreportyear"] = df_rt_ceos["annualreportdate"].dt.year
df_rt_ceos["log_totalcompensation_eur"] = np.log(
    df_rt_ceos["totalcompensation_eur"]
)

print(f"Matched CEO remuneration records: {len(df_rt_ceos):,}")
print(f"Companies with matched CEO pay: {df_rt_ceos['companyid'].nunique():,}")
print(f"CEOs with matched pay: {df_rt_ceos['directorid'].nunique():,}")

df_rt_ceos[[
    "companyname", "directorname_emp", "rolename_emp", "datestartrole",
    "dateendrole_clean", "annualreportdate", "currency",
    "totalcompensation", "totalcompensation_eur",
    "log_totalcompensation_eur", "ceo_turnover", "turnover_year"
]].head(10)

<a id="firm-year-ceo-pay-panel"></a>
### 6.4 Firm-Year CEO Pay Panel


The matched table can contain more than one CEO record for a company-year when outgoing and incoming CEOs both receive compensation, when roles overlap, or when the 90-day tolerance admits records near both spells. For this first analysis panel, we keep the latest annual-report observation within each `companyid` and `annualreportyear`. If annual-report dates tie, the current sort uses `directorid` only as a deterministic tie-breaker; that tie should not be interpreted economically.

In [ ]:
# Build a firm-year CEO pay panel from the matched remuneration records
panel_cols = [
    "companyid", "companyname", "boardid", "boardname",
    "directorid", "directorname_emp", "rolename_emp",
    "datestartrole", "dateendrole_clean",
    "annualreportdate", "annualreportyear", "currency",
    "remun_fx_rate_avg_to_eur",
    "salary", "salary_eur", "bonus", "bonus_eur",
    "totalcompensation", "totalcompensation_eur", "log_totalcompensation_eur",
    "totaldirectcomp", "totaldirectcomp_eur", "perftotal",
    "valtoteqheld", "valtoteqheld_eur",
    "valltipheld", "valltipheld_eur",
    "valeqaward", "valeqaward_eur",
    "ltipvalue", "ltipvalue_eur",
    "toteqatrisk", "toteqatrisk_eur",
    "ceo_turnover", "turnover_year"
]

available_panel_cols = [col for col in panel_cols if col in df_rt_ceos.columns]
missing_panel_cols = [col for col in panel_cols if col not in df_rt_ceos.columns]

if missing_panel_cols:
    print(f"Panel columns unavailable and omitted: {missing_panel_cols}")

ceo_pay_panel = df_rt_ceos[available_panel_cols].copy()

# Keep one record per firm-year: latest report date, then directorid as a deterministic tie-breaker.
ceo_pay_panel = (
    ceo_pay_panel
    .sort_values(["companyid", "annualreportyear", "annualreportdate", "directorid"])
    .drop_duplicates(subset=["companyid", "annualreportyear"], keep="last")
    .sort_values(["companyid", "annualreportyear"])
    .reset_index(drop=True)
)

print(f"Firm-year CEO pay observations: {len(ceo_pay_panel):,}")
print(f"Unique companies: {ceo_pay_panel['companyid'].nunique():,}")
print(f"Year range: {ceo_pay_panel['annualreportyear'].min()}-{ceo_pay_panel['annualreportyear'].max()}")

ceo_pay_panel

Next, we merge in the CEO turnover firm-year indicator created from the employment data. `ceo_turnover_dummy` equals one in years where a new CEO spell starts at the firm. Missing values are set to zero, meaning no observed CEO turnover in that firm-year.

In [ ]:
# Merge firm-year CEO turnover indicator into the pay panel
ceo_pay_panel = ceo_pay_panel.merge(
    ceo_turnover_firm_year,
    left_on=["companyid", "annualreportyear"],
    right_on=["companyid", "turnover_year"],
    how="left",
    suffixes=("", "_event")
)

ceo_pay_panel["num_ceo_turnovers"] = ceo_pay_panel["num_ceo_turnovers"].fillna(0).astype(int)
ceo_pay_panel["ceo_turnover_dummy"] = ceo_pay_panel["ceo_turnover_dummy"].fillna(0).astype(int)

# If the merge creates a second turnover-year column, keep it only as event-year information.
if "turnover_year_event" in ceo_pay_panel.columns:
    ceo_pay_panel = ceo_pay_panel.rename(columns={"turnover_year_event": "turnover_event_year"})

ceo_pay_panel[[
    "companyname", "annualreportyear", "directorname_emp", "totalcompensation_eur",
    "ceo_turnover_dummy", "num_ceo_turnovers"
]]

For event-style analysis, each company is assigned its first observed turnover year. `treated_firm` identifies companies that experience at least one observed turnover, while `event_time` measures years relative to that first turnover and `post_turnover` identifies the turnover year and later observations.

In [ ]:
first_turnover_year = (
    ceo_turnover_firm_year
    .groupby("companyid")["turnover_year"]
    .min()
    .rename("first_turnover_year")
    .reset_index()
)
print(f"Firms with an observed turnover: {len(first_turnover_year):,}")
print(
    f"Observed first-turnover range: "
    f"{first_turnover_year['first_turnover_year'].min()}-"
    f"{first_turnover_year['first_turnover_year'].max()}"
)
first_turnover_year.head()

In [ ]:
# Inspect the distribution of first observed turnover years before merging it into the pay panel.
first_turnover_year["first_turnover_year"].describe()

In [ ]:
# Create event-time variables around the first observed CEO turnover per firm.
ceo_pay_panel = ceo_pay_panel.merge(
    first_turnover_year,
    on="companyid",
    how="left"
)

ceo_pay_panel["treated_firm"] = ceo_pay_panel["first_turnover_year"].notna().astype(int)
ceo_pay_panel["event_time"] = ceo_pay_panel["annualreportyear"] - ceo_pay_panel["first_turnover_year"]
ceo_pay_panel["post_turnover"] = (
    ceo_pay_panel["treated_firm"].eq(1)
    & ceo_pay_panel["event_time"].ge(0)
).astype(int)

# Useful event-window flags for later robustness checks
ceo_pay_panel["event_window_3yr"] = ceo_pay_panel["event_time"].between(-3, 3).fillna(False).astype(int)
ceo_pay_panel["event_window_5yr"] = ceo_pay_panel["event_time"].between(-5, 5).fillna(False).astype(int)

ceo_pay_panel[[
    "companyname", "annualreportyear", "first_turnover_year", "event_time",
    "treated_firm", "post_turnover", "ceo_turnover_dummy"
]]

At this stage, compensation is comparable across currencies but remains nominal. Pay growth, logarithms, winsorization, and dispersion are deliberately postponed until after the final panel receives its CPI adjustment in Section 8.2; otherwise inflation could be mistaken for a change in CEO pay.

In [ ]:
# Sort the nominal-EUR pay panel; real pay changes are created after CPI adjustment.
ceo_pay_panel = ceo_pay_panel.sort_values(["companyid", "annualreportyear"]).copy()

ceo_pay_panel[[
    "companyname", "annualreportyear", "currency",
    "totalcompensation", "remun_fx_rate_avg_to_eur",
    "totalcompensation_eur", "ceo_turnover_dummy", "post_turnover"
]]

<a id="market-data-and-market-capitalization"></a>
## 7. Market Data and Market Capitalization

Annual fundamentals and monthly prices arrive in local currencies. We first convert annual accounting values to nominal EUR, then select one exchange per firm, convert only the retained price history to EUR, and finally attach the nearest monthly price within 31 days of each fiscal reporting date.

<a id="fundamentals-eur-conversion"></a>
### 7.1 Convert Annual Fundamentals to Nominal EUR

Balance-sheet stocks (`at`, `dltt`) use the latest daily rate on or before `datadate`. Annual flows (`sale`, `ebit`, `nicon`) use the average rate for the reporting year. Original local-currency values remain unchanged for auditing and ratio construction.


In [ ]:
# Retain the annual fundamentals needed for the final analysis panel.
fundamental_cols = [
    "gvkey", "isin", "datadate", "fyear", "curcd", "at", "sale",
    "ebit", "dltt", "nicon", "emp", "cshoi",
]
df_firm = df_firm[fundamental_cols].copy()
df_firm["datadate"] = pd.to_datetime(df_firm["datadate"], errors="coerce")
df_firm["fx_year"] = df_firm["datadate"].dt.year

# Annual-average EUR rate for income-statement flow variables.
df_firm = (
    df_firm
    .drop(columns=["fromcurd", "funda_fx_rate_avg_to_eur"], errors="ignore")
    .merge(
        fx_annual_average.rename(columns={
            "fx_rate_avg_to_eur": "funda_fx_rate_avg_to_eur"
        }),
        left_on=["curcd", "fx_year"],
        right_on=["fromcurd", "fx_year"],
        how="left",
        validate="many_to_one",
    )
    .drop(columns="fromcurd")
)

# Period-end EUR rate for balance-sheet stock variables.
df_firm = attach_daily_eur_rate(
    df_firm.drop(columns=["fx_date", "funda_fx_rate_close_to_eur"], errors="ignore"),
    date_col="datadate",
    currency_col="curcd",
    rate_col="funda_fx_rate_close_to_eur",
)

for col in ["sale", "ebit", "nicon"]:
    df_firm[f"{col}_eur"] = df_firm[col] * df_firm["funda_fx_rate_avg_to_eur"]
for col in ["at", "dltt"]:
    df_firm[f"{col}_eur"] = df_firm[col] * df_firm["funda_fx_rate_close_to_eur"]

print(f"Fundamental rows: {len(df_firm):,}")
print(f"Missing annual-average FX rates: {df_firm['funda_fx_rate_avg_to_eur'].isna().sum():,}")
print(f"Missing period-end FX rates: {df_firm['funda_fx_rate_close_to_eur'].isna().sum():,}")
df_firm[[
    "gvkey", "fyear", "curcd", "at", "at_eur",
    "sale", "sale_eur", "funda_fx_rate_close_to_eur",
]].head()

In [ ]:
# Monthly prices were retrieved once in Section 5.6; reuse that compact table here.
print(f"Monthly market records available: {len(df_market):,}")
print(f"Unique GVKEYs matched: {df_market['gvkey'].nunique():,}")
df_market.head()

**Table description:** `df_market` contains monthly Compustat security prices. Because the same `gvkey` can be listed on multiple exchanges, this table is checked before choosing one exchange per firm.


In [ ]:
missing_and_duplicate_summary(df_market)

In [ ]:
exchanges = (
    df_market["exchg"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

def fetch_exchange_codes():
    wrds_conn = get_wrds_connection()
    return wrds_conn.raw_sql("""
        SELECT exchgcd, exchgdesc
        FROM comp_na_daily_all.r_ex_codes
        WHERE exchgcd IN %(exchanges)s
    """, params={"exchanges": tuple(exchanges)})


df_exchanges = load_or_fetch(
    "compustat_exchange_codes_analysis_cols_v1",
    fetch_exchange_codes,
    refresh=REFRESH_WRDS_CACHE,
)

df_exchanges

In [ ]:
# Merge exchange names into the market data for better interpretability
df_market = df_market.merge(df_exchanges, left_on="exchg", right_on="exchgcd", how="left")
df_market

<a id="market-exchange-filter"></a>
### 7.2 Select One Exchange per Firm


**Why this filter matters:** Some companies have monthly prices from several exchanges. To avoid duplicate firm-month observations and unnecessary FX conversions, `df_market_filtered` keeps the exchange with the most complete non-missing price-date coverage for each `gvkey`. Only this retained price history is converted to EUR.


In [ ]:
# Keep one exchange per gvkey: choose the exchange with the most complete price-date coverage.
df_market = df_market.copy()

if "market_datadate" not in df_market.columns:
    df_market["market_datadate"] = pd.to_datetime(df_market["datadate"], errors="coerce")
else:
    df_market["market_datadate"] = pd.to_datetime(df_market["market_datadate"], errors="coerce")

df_market["_priced_market_datadate"] = df_market["market_datadate"].where(
    df_market["prccm"].notna()
)

exchange_coverage = (
    df_market
    .groupby(["gvkey", "exchg", "exchgdesc"], dropna=False)
    .agg(
        price_date_count=("_priced_market_datadate", "nunique"),
        price_obs_count=("prccm", "count"),
        total_rows=("market_datadate", "size"),
        first_market_datadate=("market_datadate", "min"),
        last_market_datadate=("market_datadate", "max"),
    )
    .reset_index()
)

selected_exchanges = (
    exchange_coverage
    .sort_values(
        ["gvkey", "price_date_count", "price_obs_count", "last_market_datadate", "total_rows"],
        ascending=[True, False, False, False, False],
    )
    .drop_duplicates("gvkey")
)

df_market_filtered = (
    df_market
    .merge(
        selected_exchanges[["gvkey", "exchg"]].assign(_selected_exchange=1),
        on=["gvkey", "exchg"],
        how="inner",
    )
    .drop(columns=["_priced_market_datadate", "_selected_exchange"])
    .sort_values(["gvkey", "market_datadate"])
    .reset_index(drop=True)
)

multi_exchange_gvkeys = exchange_coverage.groupby("gvkey")["exchg"].nunique().gt(1).sum()

print(f"Raw market rows:        {len(df_market):,}")
print(f"Filtered market rows:   {len(df_market_filtered):,}")
print(f"GVKEYs retained:        {df_market_filtered['gvkey'].nunique():,}")
print(f"GVKEYs with >1 exchange before filtering: {multi_exchange_gvkeys:,}")

display(selected_exchanges.head(10))
df_market_filtered


In [ ]:
print("Currencies retained after exchange selection:")
sorted(df_market_filtered["curcdm"].dropna().unique())

<a id="market-eur-merge"></a>
### 7.3 Convert Market Prices and Merge with Fundamentals


**Table description:** The selected monthly closing price is converted using the latest daily FX rate on or before `market_datadate`. `df_funda` then attaches the nearest monthly observation within 31 days of each fiscal reporting date. This is a nearest-date match, not a monthly-price average. Both local-currency and nominal-EUR market capitalization are retained.


In [ ]:
# Convert only the retained monthly price histories to nominal EUR.
df_market_filtered = attach_daily_eur_rate(
    df_market_filtered.drop(columns=["fx_date", "market_fx_rate_to_eur"], errors="ignore"),
    date_col="market_datadate",
    currency_col="curcdm",
    rate_col="market_fx_rate_to_eur",
)
df_market_filtered["prccm_eur"] = (
    df_market_filtered["prccm"] * df_market_filtered["market_fx_rate_to_eur"]
)

# Make sure dates are datetime
df_firm["datadate"] = pd.to_datetime(df_firm["datadate"])
df_market_filtered["market_datadate"] = pd.to_datetime(df_market_filtered["market_datadate"])

# Make sure gvkey has the same type/format in both datasets
df_firm["gvkey"] = df_firm["gvkey"].astype(str).str.strip()
df_market_filtered["gvkey"] = df_market_filtered["gvkey"].astype(str).str.strip()

# IMPORTANT: for merge_asof, sort by date first, then gvkey
df_firm_sorted = df_firm.sort_values(["datadate", "gvkey"]).reset_index(drop=True)
df_market_sorted = df_market_filtered.sort_values(["market_datadate", "gvkey"]).reset_index(drop=True)

df_funda = pd.merge_asof(
    df_firm_sorted,
    df_market_sorted,
    left_on="datadate",
    right_on="market_datadate",
    by="gvkey",
    direction="nearest",
    tolerance=pd.Timedelta("31 days")
)

# Shares outstanding are not monetary, so the same share count scales both prices.
df_funda["market_cap_local"] = df_funda["prccm"] * df_funda["cshoi"]
df_funda["market_cap_eur"] = df_funda["prccm_eur"] * df_funda["cshoi"]

print(f"Matched monthly prices: {df_funda['prccm'].notna().sum():,}")
print(f"Matched EUR market prices: {df_funda['prccm_eur'].notna().sum():,}")
df_funda

---

<a id="final-analysis-panel"></a>
## 8. Final Analysis Panel

`ceo_pay_panel` is the firm-year backbone. All monetary inputs have already been converted to nominal EUR before this merge. Fundamentals and market capitalization are attached by `isin` and `annualreportyear`; executive age and gender by `directorid`; and sector and headquarters country through a de-duplicated company bridge. The merge preserves one observation per company-year. CPI adjustment is applied only after these inputs are consolidated.


<a id="construct-final-panel"></a>
### 8.1 Merge the Nominal-EUR Analysis Inputs


In [ ]:
FUNDA_COLS = [
    "gvkey", "isin", "datadate_x", "fyear", "curcd", "curcdm",
    "funda_fx_rate_avg_to_eur", "funda_fx_rate_close_to_eur",
    "market_fx_rate_to_eur", "at", "at_eur", "sale", "sale_eur",
    "ebit", "ebit_eur", "dltt", "dltt_eur",
    "nicon", "nicon_eur", "emp", "cshoi",
    "prccm", "prccm_eur", "market_cap_local", "market_cap_eur",
]
EXEC_COLS = ["directorid", "age", "gender"]
CEO_PAY_COLS = list(ceo_pay_panel.columns)


In [ ]:
# Merge CEO pay, firm fundamentals, market cap, and executive demographics into one final panel.
funda_cols_available = [col for col in FUNDA_COLS if col in df_funda.columns]
exec_cols_available = [col for col in EXEC_COLS if col in df_exec.columns]

ceo_panel_for_merge = ceo_pay_panel[list(CEO_PAY_COLS)].copy()

if "isin" not in ceo_panel_for_merge.columns:
    firm_isin_bridge = (
        df_emp[["companyid", "isin"]]
        .dropna(subset=["companyid", "isin"])
        .drop_duplicates(subset=["companyid"])
    )
    ceo_panel_for_merge = ceo_panel_for_merge.merge(
        firm_isin_bridge,
        on="companyid",
        how="left",
    )

df_funda_final = (
    df_funda[funda_cols_available]
    .rename(columns={
        "datadate_x": "datadate",
        "fyear": "annualreportyear",
    })
)

if "isin" not in ceo_panel_for_merge.columns or "isin" not in df_funda_final.columns:
    raise KeyError("No ISIN identifier available to merge ceo_pay_panel with df_funda_final.")

merge_keys = ["isin", "annualreportyear"]

df_funda_final = df_funda_final.drop_duplicates(subset=merge_keys)

df_exec_final = (
    df_exec[exec_cols_available]
    .drop_duplicates(subset=["directorid"])
)

# Build a one-row-per-company bridge for sector and headquarters country.
# Prefer the most complete context row when a company has several employment records.
firm_context = (
    df_emp[["companyid", "sector", "hocountryname"]]
    .dropna(subset=["companyid"])
    .assign(
        _missing_context=lambda x: x[["sector", "hocountryname"]]
        .isna()
        .sum(axis=1)
    )
    .sort_values(["companyid", "_missing_context"])
    .drop_duplicates(subset=["companyid"], keep="first")
    .drop(columns="_missing_context")
)

df_panel_final = (
    ceo_panel_for_merge
    .merge(df_funda_final, on=merge_keys, how="left")
    .merge(df_exec_final, on="directorid", how="left")
    .merge(firm_context, on="companyid", how="left")
)

print(f"Merge keys:              {merge_keys}")
print(f"CEO pay panel rows:      {len(ceo_pay_panel):,}")
print(f"Final panel rows:        {len(df_panel_final):,}")
print(f"Final panel columns:     {df_panel_final.shape[1]:,}")
print(f"Matched fundamentals:    {df_panel_final['gvkey'].notna().sum():,}")
print(f"Matched executive data:  {df_panel_final['age'].notna().sum():,}")
print(f"Matched sector data:     {df_panel_final['sector'].notna().sum():,}")

df_panel_final.head()


In [ ]:
# Ratios are currency-invariant, so compute them from same-currency reported values.
df_panel_final["roa"] = df_panel_final["nicon"] / df_panel_final["at"].replace(0, np.nan)
df_panel_final["ebit_margin"] = df_panel_final["ebit"] / df_panel_final["sale"].replace(0, np.nan)
df_panel_final["leverage"] = df_panel_final["dltt"] / df_panel_final["at"].replace(0, np.nan)
df_panel_final["log_assets_eur_nominal"] = np.log(
    df_panel_final["at_eur"].replace(0, np.nan)
)
df_panel_final["log_market_cap_eur_nominal"] = np.log(
    df_panel_final["market_cap_eur"].replace(0, np.nan)
)

df_panel_final[[
    "isin", "annualreportyear", "roa", "ebit_margin", "leverage",
    "log_assets_eur_nominal", "log_market_cap_eur_nominal",
]].head()


In [ ]:
df_panel_final

In [ ]:
# Keep identifiers, treatment timing, outcomes, and auditable firm controls.
final_cols = [
    "isin", "gvkey", "companyid", "companyname", "sector", "hocountryname",
    "directorid", "directorname_emp", "gender", "age", "rolename_emp",
    "datestartrole", "dateendrole_clean", "annualreportdate", "annualreportyear",
    "currency", "curcd", "curcdm",
    "remun_fx_rate_avg_to_eur", "funda_fx_rate_avg_to_eur",
    "funda_fx_rate_close_to_eur", "market_fx_rate_to_eur",
    "salary", "salary_eur", "bonus", "bonus_eur",
    "totalcompensation", "totalcompensation_eur",
    "log_totalcompensation_eur", "totaldirectcomp", "totaldirectcomp_eur",
    "ceo_turnover", "turnover_year", "turnover_event_year", "num_ceo_turnovers",
    "ceo_turnover_dummy", "first_turnover_year", "treated_firm", "event_time",
    "post_turnover", "event_window_3yr", "event_window_5yr",
    "at", "at_eur", "sale", "sale_eur", "ebit", "ebit_eur",
    "dltt", "dltt_eur", "nicon", "nicon_eur", "emp", "cshoi",
    "prccm", "prccm_eur", "market_cap_local", "market_cap_eur",
    "roa", "ebit_margin", "leverage",
    "log_assets_eur_nominal", "log_market_cap_eur_nominal",
]

missing_final_cols = [col for col in final_cols if col not in df_panel_final.columns]
if missing_final_cols:
    raise KeyError(f"Expected final-panel columns are missing: {missing_final_cols}")

df_panel_final = df_panel_final[final_cols].copy()
df_panel_final

In [ ]:
# Confirm that the final table still has exactly one observation per firm-year.
duplicate_firm_years = df_panel_final.duplicated(
    subset=["companyid", "annualreportyear"],
    keep=False,
).sum()

print(f"Final panel shape: {df_panel_final.shape}")
print(f"Unique companies: {df_panel_final['companyid'].nunique():,}")
print(f"Duplicate firm-year rows: {duplicate_firm_years:,}")

if duplicate_firm_years:
    raise ValueError("The final panel is no longer unique by company-year.")

df_panel_final = df_panel_final.sort_values(
    ["companyid", "annualreportyear"]
).reset_index(drop=True)
df_panel_final

In [ ]:
missing_and_duplicate_summary(df_panel_final)

In [ ]:
df_panel_final.annualreportdate.min()

In [ ]:
df_panel_final.annualreportdate.max()

---

<a id="cpi-adjustment"></a>
### 8.2 Convert Nominal EUR to Real 2015 EUR

Currency conversion makes monetary values comparable across countries at a given date, but it does not remove inflation over time. We therefore merge annual World Bank CPI by headquarters country and report year, rebase each country to 2015 = 100, and deflate the nominal-EUR variables. Logs and within-firm pay changes are created only after this step.

**Interpretation:** a real-EUR value measures purchasing value in 2015 prices. Original local-currency and nominal-EUR columns remain in the panel, so every transformation is auditable. Rows outside CPI coverage (currently before 2000 or after 2024) remain missing rather than being extrapolated.

In [ ]:
# Validate that every currency entering the panel has an EUR conversion series.
required_fx_currencies = (
    set(df_panel_final["currency"].dropna().astype(str))
    | set(df_panel_final["curcd"].dropna().astype(str))
    | set(df_panel_final["curcdm"].dropna().astype(str))
)
available_fx_currencies = set(df_gcurrency["fromcurd"].dropna().astype(str))
missing_fx_currencies = sorted(required_fx_currencies - available_fx_currencies)

print(f"Currencies required by final panel: {sorted(required_fx_currencies)}")
print(f"Missing EUR conversion series: {missing_fx_currencies}")
if missing_fx_currencies:
    raise ValueError("Complete FX coverage is required before CPI adjustment.")

In [ ]:
# Map BoardEx headquarters-country names to World Bank ISO-2 codes.
COUNTRY_ISO2 = {
    "Austria": "AT", "Belgium": "BE", "Denmark": "DK",
    "Finland": "FI", "France": "FR", "Germany": "DE",
    "Ireland": "IE", "Republic Of Ireland": "IE", "Italy": "IT",
    "Luxembourg": "LU", "Netherlands": "NL", "Norway": "NO",
    "Poland": "PL", "Portugal": "PT", "Spain": "ES",
    "Sweden": "SE", "Switzerland": "CH", "United Kingdom": "GB",
    "Isle Of Man": "GB", "Jersey": "GB", "United States": "US",
}

df_panel_final["iso2"] = df_panel_final["hocountryname"].map(COUNTRY_ISO2)
uk_variant = (
    df_panel_final["iso2"].isna()
    & df_panel_final["hocountryname"].str.startswith("United Kingdom", na=False)
)
df_panel_final.loc[uk_variant, "iso2"] = "GB"

unmapped_countries = sorted(
    df_panel_final.loc[df_panel_final["iso2"].isna(), "hocountryname"]
    .dropna()
    .unique()
)
print(f"Unmapped headquarters countries: {unmapped_countries}")

In [ ]:
# Load cached World Bank CPI and rebase every country to 2015 = 100.
CPI_FILE = DATA_DIR / "cpi_country_annual.csv"
df_cpi = pd.read_csv(CPI_FILE)
df_cpi["year"] = pd.to_numeric(df_cpi["year"], errors="coerce").astype("Int64")
df_cpi["cpi"] = pd.to_numeric(df_cpi["cpi"], errors="coerce")

cpi_2015 = (
    df_cpi[df_cpi["year"].eq(2015)]
    .set_index("iso2")["cpi"]
    .rename("cpi_2015")
)
df_cpi = df_cpi.join(cpi_2015, on="iso2")
df_cpi["cpi_index_2015"] = df_cpi["cpi"] / df_cpi["cpi_2015"] * 100

print(f"CPI countries: {df_cpi['iso2'].nunique():,}")
print(f"CPI year range: {df_cpi['year'].min()}-{df_cpi['year'].max()}")

In [ ]:
# Attach country-year CPI without extrapolating beyond observed coverage.
df_panel_final = (
    df_panel_final
    .drop(columns=["year", "cpi", "cpi_2015", "cpi_index_2015"], errors="ignore")
    .merge(
        df_cpi[["iso2", "year", "cpi", "cpi_2015", "cpi_index_2015"]],
        left_on=["iso2", "annualreportyear"],
        right_on=["iso2", "year"],
        how="left",
        validate="many_to_one",
    )
    .drop(columns="year")
)

print(f"Rows with CPI coverage: {df_panel_final['cpi_index_2015'].notna().sum():,} / {len(df_panel_final):,}")

In [ ]:
# Deflate nominal EUR values into country-specific 2015 purchasing values.
nominal_to_real = {
    "salary_eur": "real_salary_eur_2015",
    "bonus_eur": "real_bonus_eur_2015",
    "totalcompensation_eur": "real_totalcompensation_eur_2015",
    "totaldirectcomp_eur": "real_totaldirectcomp_eur_2015",
    "at_eur": "real_at_eur_2015",
    "sale_eur": "real_sale_eur_2015",
    "ebit_eur": "real_ebit_eur_2015",
    "dltt_eur": "real_dltt_eur_2015",
    "nicon_eur": "real_nicon_eur_2015",
    "market_cap_eur": "real_market_cap_eur_2015",
}

for nominal_col, real_col in nominal_to_real.items():
    df_panel_final[real_col] = (
        df_panel_final[nominal_col]
        / df_panel_final["cpi_index_2015"]
        * 100
    )

In [ ]:
# Create modelling variables only after both FX and CPI adjustment.
df_panel_final = df_panel_final.sort_values(
    ["companyid", "annualreportyear"]
).copy()

df_panel_final["log_real_totalcompensation_eur_2015"] = np.log(
    df_panel_final["real_totalcompensation_eur_2015"].replace(0, np.nan)
)
df_panel_final["log_real_assets_eur_2015"] = np.log(
    df_panel_final["real_at_eur_2015"].replace(0, np.nan)
)
df_panel_final["log_real_market_cap_eur_2015"] = np.log(
    df_panel_final["real_market_cap_eur_2015"].replace(0, np.nan)
)

df_panel_final["lag_real_totalcompensation_eur_2015"] = (
    df_panel_final.groupby("companyid")["real_totalcompensation_eur_2015"].shift(1)
)
df_panel_final["real_pay_change_eur_2015"] = (
    df_panel_final["real_totalcompensation_eur_2015"]
    - df_panel_final["lag_real_totalcompensation_eur_2015"]
)
df_panel_final["real_pay_change_pct"] = (
    df_panel_final["real_pay_change_eur_2015"]
    / df_panel_final["lag_real_totalcompensation_eur_2015"].replace(0, np.nan)
)
df_panel_final["log_real_pay_change"] = (
    df_panel_final.groupby("companyid")["log_real_totalcompensation_eur_2015"].diff()
)

In [ ]:
real_columns = list(nominal_to_real.values())
real_coverage = pd.DataFrame({
    "non_null": df_panel_final[real_columns].notna().sum(),
    "missing": df_panel_final[real_columns].isna().sum(),
    "coverage_pct": (df_panel_final[real_columns].notna().mean() * 100).round(2),
})

years_without_cpi = sorted(
    df_panel_final.loc[df_panel_final["cpi_index_2015"].isna(), "annualreportyear"]
    .dropna()
    .unique()
)
print(f"Report years without CPI coverage: {years_without_cpi}")
real_coverage

In [ ]:
print("Insights:")
print("- Monetary variables are now comparable across currencies and years.")
print("- Financial ratios are unchanged by currency conversion because their components share a currency.")
print("- Pay growth now reflects real purchasing-value changes rather than inflation.")
print(f"- Final analysis panel: {df_panel_final.shape[0]:,} rows x {df_panel_final.shape[1]:,} columns.")
print(
    "- Duplicate firm-years after CPI merge:",
    df_panel_final.duplicated(["companyid", "annualreportyear"]).sum(),
)

df_panel_final[[
    "companyname", "annualreportyear", "hocountryname", "currency",
    "totalcompensation", "totalcompensation_eur",
    "cpi_index_2015", "real_totalcompensation_eur_2015",
    "log_real_totalcompensation_eur_2015", "real_pay_change_pct",
]].head(10)

<a id="winsorization"></a>
### 8.3 Winsorization of Pay (p1 / p99)

Nominal outliers and extreme values are clipped at the 1st and 99th percentile of
`real_totalcompensation_eur_2015`. The log of the winsorized value becomes the primary
regression outcome (`log_w_real_pay`), following the convention established in the
causal-inference and supervised-learning sections.

In [ ]:
# --- 8.3  Winsorize real compensation at p1/p99 (among non-null rows) ---
def winsorize_series(s, lo=0.01, hi=0.99):
    """Clip to [p1, p99]; NaN-safe."""
    q_lo, q_hi = s.quantile([lo, hi])
    return s.clip(lower=q_lo, upper=q_hi)

real_pay_mapping = {
    "real_totalcompensation_eur_2015": "w_real_totalcompensation_eur_2015",
    "real_salary_eur_2015":            "w_real_salary_eur_2015",
    "real_bonus_eur_2015":             "w_real_bonus_eur_2015",
}
for src_col, dst_col in real_pay_mapping.items():
    if src_col in df_panel_final.columns:
        df_panel_final[dst_col] = winsorize_series(df_panel_final[src_col])

# Log of winsorized total pay — primary regression outcome
df_panel_final["log_w_real_pay"] = np.log(
    df_panel_final["w_real_totalcompensation_eur_2015"].replace(0, np.nan)
)

stats = df_panel_final[[
    "real_totalcompensation_eur_2015",
    "w_real_totalcompensation_eur_2015",
    "log_w_real_pay",
]].describe()
display(stats)

p1  = df_panel_final["real_totalcompensation_eur_2015"].quantile(0.01)
p99 = df_panel_final["real_totalcompensation_eur_2015"].quantile(0.99)
n_lo = (df_panel_final["real_totalcompensation_eur_2015"] < p1).sum()
n_hi = (df_panel_final["real_totalcompensation_eur_2015"] > p99).sum()
print(f"\np1={p1:,.0f}, p99={p99:,.0f}  (2015 EUR)")
print(f"Rows clipped at lower tail: {n_lo}")
print(f"Rows clipped at upper tail: {n_hi}")

<a id="save-panel"></a>
### 8.4 Save Clean Panel

Both `panel_clean.csv` and `panel_analysis.csv` are written from `df_panel_final`.
Convenience alias columns (`year`, `log_assets`, `ceo_turnover`) are added before
saving so that the causal-inference section (13.1) can reference them directly.

In [ ]:
# --- 8.4  Persist the clean analysis panel ---

# Convenience aliases used in causal inference and supervised learning sections
df_panel_final["year"]         = df_panel_final["annualreportyear"]
df_panel_final["log_assets"]   = np.log(
    df_panel_final["real_at_eur_2015"].replace(0, np.nan)
)
# ceo_turnover_dummy was built in Section 6.4; rename to shorter alias
if "ceo_turnover_dummy" in df_panel_final.columns:
    df_panel_final["ceo_turnover"] = df_panel_final["ceo_turnover_dummy"].fillna(0).astype(int)
else:
    df_panel_final["ceo_turnover"] = 0

PANEL_CLEAN    = DATA_DIR / "panel_clean.csv"
PANEL_ANALYSIS = DATA_DIR / "panel_analysis.csv"

df_panel_final.to_csv(PANEL_CLEAN,    index=False)
df_panel_final.to_csv(PANEL_ANALYSIS, index=False)

print(f"Saved: {PANEL_CLEAN}")
print(f"Saved: {PANEL_ANALYSIS}")
print(f"Shape: {df_panel_final.shape}")
print(f"Columns ({len(df_panel_final.columns)}): {df_panel_final.columns.tolist()}")

---

<a id="data-visualizations"></a>
## 9. Data Visualizations

This section visualizes the analysis-ready firm-year panel. Compensation is shown in **thousands of EUR at 2015 prices**, matching the units of the BoardEx remuneration fields after currency and CPI adjustment. The figures focus on the main outcome, its evolution over time and around CEO turnover, and the firm/executive characteristics considered as exploratory controls.

<a id="target-distribution"></a>
### 9.1 Target Distribution

The first figure compares real compensation in levels with its logarithmic transformation. The level plot preserves the economic scale, while the log plot makes the dense center of the distribution easier to inspect and matches the baseline modelling outcome.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

viz_df = df_panel_final.copy()
viz_df = viz_df.replace([np.inf, -np.inf], np.nan)

target_level = "real_totalcompensation_eur_2015"
target_log = "log_real_totalcompensation_eur_2015"
target_plot = viz_df[[target_level, target_log]].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(
    data=target_plot,
    x=target_level,
    bins=35,
    kde=True,
    color="#2A6F97",
    ax=axes[0],
)
axes[0].axvline(
    target_plot[target_level].median(),
    color="#C44536",
    linestyle="--",
    label="Median",
)
axes[0].set(
    title="Real CEO compensation in levels",
    xlabel="Total compensation (EUR thousands, 2015 prices)",
    ylabel="Firm-year observations",
)
axes[0].legend()

sns.histplot(
    data=target_plot,
    x=target_log,
    bins=35,
    kde=True,
    color="#4C956C",
    ax=axes[1],
)
axes[1].axvline(
    target_plot[target_log].median(),
    color="#C44536",
    linestyle="--",
    label="Median",
)
axes[1].set(
    title="Log real CEO compensation",
    xlabel="log(1 + real total compensation)",
    ylabel="Firm-year observations",
)
axes[1].legend()

plt.tight_layout()
plt.show()

**Insights:**

- Real total compensation is right-skewed: the median is about **EUR 2.04 million**, the mean is **EUR 2.30 million**, and the 99th percentile is about **EUR 7.44 million**.
    
- The log transformation compresses the high-pay tail and is therefore more suitable as the baseline regression target. A small number of unusually low observations still create a left tail after logging, so later models should report sensitivity to winsorization or explicit minimum-pay filters.

<a id="annual-pay-trend"></a>
### 9.2 Annual Pay Trend and Coverage

The next figure separates the evolution of real pay from the number of usable observations. Mean and median compensation are both shown because their gap reveals whether a small high-pay tail is influencing the annual average.

In [ ]:
annual_pay = (
    viz_df.dropna(subset=["annualreportyear", target_level])
    .groupby("annualreportyear", as_index=False)
    .agg(
        median_real_pay=(target_level, "median"),
        mean_real_pay=(target_level, "mean"),
        observations=(target_level, "size"),
    )
)

fig, axes = plt.subplots(
    1, 2, figsize=(14, 4.5),
    gridspec_kw={"width_ratios": [2, 1]},
)

sns.lineplot(
    data=annual_pay,
    x="annualreportyear",
    y="median_real_pay",
    marker="o",
    label="Median",
    color="#2A6F97",
    ax=axes[0],
)
sns.lineplot(
    data=annual_pay,
    x="annualreportyear",
    y="mean_real_pay",
    marker="o",
    label="Mean",
    color="#C44536",
    ax=axes[0],
)
axes[0].set(
    title="Real CEO compensation over time",
    xlabel="Annual report year",
    ylabel="EUR thousands (2015 prices)",
)

sns.barplot(
    data=annual_pay,
    x="annualreportyear",
    y="observations",
    color="#84A98C",
    ax=axes[1],
)
axes[1].set(
    title="Usable pay observations by year",
    xlabel="Annual report year",
    ylabel="Observations",
)
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

**Insights**

- Median real compensation rises from approximately **EUR 1.31 million in 2000** to a sample peak of **EUR 2.58 million in 2017**, before easing to about **EUR 1.95 million in 2024**.

- The mean is above the median in most years, consistent with the right-skewed target distribution. Coverage also expands from **26 observations in 2000** to **112 in 2024**.

<a id="turnover-event-visualization"></a>
### 9.3 Pay Around CEO Turnover

For treated firms, event time equals zero in the first observed CEO-turnover year. The line shows median real compensation from five years before through five years after that event; the bars show how many firm-years support each point.

In [ ]:
event_pay = (
    viz_df.loc[
        viz_df["treated_firm"].eq(1)
        & viz_df["event_time"].between(-5, 5)
        & viz_df[target_level].notna()
    ]
    .groupby("event_time", as_index=False)
    .agg(
        median_real_pay=(target_level, "median"),
        observations=(target_level, "size"),
    )
)
event_pay["event_time"] = event_pay["event_time"].astype(int)

fig, ax_pay = plt.subplots(figsize=(11, 5))
ax_count = ax_pay.twinx()

ax_count.bar(
    event_pay["event_time"],
    event_pay["observations"],
    color="#B8C0CC",
    alpha=0.45,
    label="Observations",
)
ax_pay.plot(
    event_pay["event_time"],
    event_pay["median_real_pay"],
    color="#2A6F97",
    marker="o",
    linewidth=2.5,
    label="Median real pay",
)
ax_pay.axvline(0, color="#C44536", linestyle="--", linewidth=1.5)
ax_pay.set(
    title="Median real CEO compensation around the first turnover",
    xlabel="Years relative to first CEO turnover",
    ylabel="Median compensation (EUR thousands, 2015 prices)",
)
ax_count.set_ylabel("Firm-year observations")
ax_pay.set_xticks(range(-5, 6))

handles_1, labels_1 = ax_pay.get_legend_handles_labels()
handles_2, labels_2 = ax_count.get_legend_handles_labels()
ax_pay.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left")

plt.tight_layout()
plt.show()

**Insights:**

- Median real compensation falls from about **EUR 2.07 million in event year -1** to **EUR 1.41 million in the turnover year**, then rebounds to approximately **EUR 2.18 million in year +1**.

<a id="exploratory-relationships"></a>
### 9.4 Exploratory Relationships

The correlation heatmap summarizes continuous candidate controls. The sector boxplots use the eight sectors with the most usable pay observations and suppress individual outlier markers so that differences in their central distributions remain visible.

In [ ]:
correlation_columns = {
    "log_real_totalcompensation_eur_2015": "Log real pay",
    "log_real_assets_eur_2015": "Log real assets",
    "log_real_market_cap_eur_2015": "Log real market cap",
    "roa": "ROA",
    "ebit_margin": "EBIT margin",
    "leverage": "Leverage",
    "age": "CEO age",
}

corr_data = (
    viz_df[list(correlation_columns)]
    .apply(pd.to_numeric, errors="coerce")
    .rename(columns=correlation_columns)
)

sector_plot = viz_df.dropna(subset=["sector", target_log]).copy()
top_sectors = sector_plot["sector"].value_counts().head(8).index
sector_plot = sector_plot[sector_plot["sector"].isin(top_sectors)]
sector_order = (
    sector_plot.groupby("sector")[target_log]
    .median()
    .sort_values(ascending=False)
    .index
)

fig, axes = plt.subplots(
    1, 2, figsize=(15, 6),
    gridspec_kw={"width_ratios": [1, 1.2]},
)

sns.heatmap(
    corr_data.corr(),
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=axes[0],
)
axes[0].set_title("Correlations among target and exploratory variables")

sns.boxplot(
    data=sector_plot,
    x=target_log,
    y="sector",
    order=sector_order,
    color="#84A98C",
    showfliers=False,
    ax=axes[1],
)
axes[1].set(
    title="Log real compensation in the most represented sectors",
    xlabel="log(1 + real total compensation)",
    ylabel="Sector",
)

plt.tight_layout()
plt.show()

**Insights:**

- Log real pay has modest positive correlations with **log market capitalization (0.27)** and **log assets (0.24)**, while its correlations with profitability, leverage, and CEO age are weaker in the pooled sample.

- The stronger correlations between assets and market capitalization (**0.56**) and between ROA and EBIT margin (**0.69**) warn against adding highly related controls without multicollinearity checks.

- Sector distributions also differ visibly—among the most represented sectors, clothing/personal-products and oil-and-gas observations tend to sit above banking, telecommunications, leisure/hotels, and information-technology hardware. These are pooled associations, not estimates of within-firm or causal effects.

- **Modelling implication.** Use log real compensation as the main target, retain firm size and sector as important explanatory dimensions, and estimate turnover effects with firm/year controls rather than reading them directly from the descriptive plots.

<a id="grouped-pay-comparisons"></a>
### 9.5 Mean and Median Pay Across CEO and Firm Groups

Mean and median real CEO compensation are plotted together because their distance is itself informative: a mean materially above the median indicates that a small high-pay tail is pulling up the average. Age and gender include every observed group. To keep labels readable, the industry and headquarters-country panels visualize the **12 groups with the most usable compensation observations**; the complete results remain available in `grouped_ceo_pay_summary`. Each marker is based on firm-year observations, not unique CEOs. The age bands use the BoardEx `age` field retained in the panel; because date of birth is not retained here, this may not equal age in each historical report year.


In [ ]:
# Construct age bands and harmonize the gender labels used in the grouped summaries.
grouped_viz_df = viz_df.copy()
grouped_viz_df["age_numeric"] = pd.to_numeric(grouped_viz_df["age"], errors="coerce")
grouped_viz_df["age_group"] = pd.cut(
    grouped_viz_df["age_numeric"],
    bins=[0, 39, 49, 59, 69, np.inf],
    labels=["Under 40", "40–49", "50–59", "60–69", "70+"],
)

grouped_viz_df["gender_group"] = (
    grouped_viz_df["gender"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({"MALE": "M", "FEMALE": "F"})
)

def summarize_grouped_pay(data, group_column, dimension):
    """Return mean, median, and coverage for one grouping variable."""
    summary = (
        data.dropna(subset=[group_column, target_level])
        .groupby(group_column, observed=True)[target_level]
        .agg(mean_pay="mean", median_pay="median", observations="size")
        .reset_index()
        .rename(columns={group_column: "group"})
    )
    summary.insert(0, "dimension", dimension)
    summary["group"] = summary["group"].astype(str)
    return summary

age_pay_summary = summarize_grouped_pay(grouped_viz_df, "age_group", "Age group")
gender_pay_summary = summarize_grouped_pay(grouped_viz_df, "gender_group", "Gender")
industry_pay_summary = summarize_grouped_pay(grouped_viz_df, "sector", "Industry")
country_pay_summary = summarize_grouped_pay(
    grouped_viz_df, "hocountryname", "Headquarters country"
)

grouped_ceo_pay_summary = pd.concat(
    [
        age_pay_summary,
        gender_pay_summary,
        industry_pay_summary,
        country_pay_summary,
    ],
    ignore_index=True,
)

def most_observed(summary, number=12):
    return summary.nlargest(number, "observations").sort_values(
        "median_pay", ascending=True
    )

plot_summaries = {
    "Age group": age_pay_summary.iloc[::-1],
    "Gender": gender_pay_summary.sort_values("median_pay"),
    "Industry (12 most observed)": most_observed(industry_pay_summary),
    "Country (12 most observed)": most_observed(country_pay_summary),
}

def draw_mean_median_dumbbell(ax, summary, title):
    """Plot group medians and means on the same horizontal scale."""
    positions = np.arange(len(summary))
    lower = summary[["mean_pay", "median_pay"]].min(axis=1)
    upper = summary[["mean_pay", "median_pay"]].max(axis=1)

    ax.hlines(
        positions,
        lower,
        upper,
        color="#AAB2BD",
        linewidth=2,
        zorder=1,
    )
    ax.scatter(
        summary["median_pay"],
        positions,
        color="#2A6F97",
        s=55,
        label="Median",
        zorder=2,
    )
    ax.scatter(
        summary["mean_pay"],
        positions,
        color="#C44536",
        marker="D",
        s=48,
        label="Mean",
        zorder=2,
    )
    ax.set_yticks(positions)
    ax.set_yticklabels(summary["group"])
    ax.set_title(title)
    ax.set_xlabel("Real CEO compensation (EUR thousands, 2015 prices)")
    ax.set_ylabel("")

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
for ax, (title, summary) in zip(axes.flat, plot_summaries.items()):
    draw_mean_median_dumbbell(ax, summary, title)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.955),
    ncol=2,
    frameon=False,
)
fig.suptitle(
    "Mean and median real CEO compensation across groups",
    fontsize=15,
    y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.show()

display(
    grouped_ceo_pay_summary.assign(
        mean_pay_million_eur=lambda x: x["mean_pay"] / 1_000,
        median_pay_million_eur=lambda x: x["median_pay"] / 1_000,
    )[
        [
            "dimension",
            "group",
            "mean_pay_million_eur",
            "median_pay_million_eur",
            "observations",
        ]
    ].round(2)
)


**Insights:**

- Recorded mean pay increases from approximately **EUR 1.77 million for ages 40–49** to **EUR 2.38 million for ages 70+**, although the 40–49 group contains only **26 firm-years**.

- Female CEO observations have mean/median pay of approximately **EUR 1.71/1.49 million**, compared with **EUR 2.31/2.08 million** for male observations; this comparison is highly unbalanced (**54 versus 1,960 firm-years**) and is not an adjusted gender pay gap.

- Among the 12 most represented industries, median pay ranges from roughly **EUR 1.52 million** in leisure/hotels to **EUR 3.10 million** in clothing/personal products. Across the displayed countries, it ranges from about **EUR 0.82 million** in Norway to **EUR 2.86 million** in Italy.

- The gaps between paired markers also reveal within-group skewness—for example, banking and Finland have means substantially above their medians. All comparisons are pooled and descriptive: they combine firm size, sector, country, year, and repeated CEO observations. Formal age, gender, industry, or country effects therefore require multivariate modelling and appropriate firm/year controls.


---

<a id="planned-methods"></a>
## 10. Planned Methods


The completed preparation pipeline produces `df_panel_final` with raw values, nominal EUR values, and real 2015 EUR values. Modelling should use `log_real_totalcompensation_eur_2015` as the baseline pay outcome and `log_real_assets_eur_2015` or `log_real_market_cap_eur_2015` as size controls. The next stage applies one method from each course block, with a shared emphasis on transparent preprocessing and robustness evaluation.

### 10.1 Causal Inference
- [X] Causal graph / DAG (DoWhy)
- [X] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:* We will model CEO turnover as a governance shock in a DAG and use backdoor adjustment to control for firm fundamentals that affect both turnover and pay. We will estimate the causal effect of turnover using pre/post (event‑study style) comparisons around the turnover year.

### 10.2 Supervised Learning
- [X] Linear / Ridge / Lasso regression
- [X] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [X] Gradient Boosting (XGBoost / LightGBM / sklearn GBM)
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification:* Supervised models will benchmark expected compensation conditional on firm and executive characteristics. Linear models provide interpretable baselines and covariate effects, while tree-based and boosting models capture nonlinearities/interactions for more accurate counterfactual pay predictions. We will use Optuna to tune boosting hyperparameters (e.g., depth, learning rate, subsampling) to avoid overfitting and compare against simpler baselines.

### 10.3 Unsupervised Learning / Generative Models
- [X] K-Means clustering
- [ ] Hierarchical clustering
- [X] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* Clustering will segment firms into comparable peer groups before causal estimation and highlight heterogeneous effects across turnover regimes. A VAE will learn low-dimensional representations of firm/executive profiles to detect anomalous pay structures and support exploratory subgroup analysis.

<a id="evaluation-strategy"></a>
## 11. Evaluation Strategy


The mission succeeds if the final notebook produces a transparent, reproducible panel and uses it to answer the research question from three angles:

- **Causal inference:** estimate whether CEO turnover is associated with changes in real 2015 EUR CEO pay after adjusting for firm characteristics.
- **Prediction:** benchmark models of real 2015 EUR compensation using RMSE/MAE and classification metrics where turnover or high-pay outcomes are modeled.
- **Unsupervised learning:** identify firm or executive clusters that reveal heterogeneous compensation patterns.
- **Robustness:** report missing-data coverage, sensitivity to winsorization, and whether results change across event windows.


<a id="work-plan"></a>
## 12. Work Plan


| Step | Owner | Description | Output | Status |
|---:|---|---|---|---|
| 1 | Achmad | Define the STOXX 600 universe and retrieve BoardEx/Compustat inputs | Raw source tables | Complete |
| 2 | Kajetan | Clean identifiers, dates, compensation variables, and missing values | Harmonized source tables | Complete |
| 3 | Achmad | Identify firm-level CEO spells and turnover events | `ceo_turnover_firm_year` | Complete |
| 4 | Kajetan | Match CEO remuneration to valid CEO spells | `ceo_pay_panel` | Complete |
| 5 | Achmad | Convert monetary inputs to EUR; add market capitalization and financial ratios | Nominal-EUR source tables and `df_funda` | Complete |
| 6 | Achmad + Kajetan | Merge inputs, apply CPI, and validate firm-year uniqueness | Real-2015-EUR `df_panel_final` | Complete |
| 7 | Kajetan | Implement causal and event-window analysis | Estimates and robustness checks | Complete |
| 8 | Achmad | Implement supervised and unsupervised learning | Model metrics, clusters, and heterogeneity | Planned |
| 9 | Achmad + Kajetan | Synthesize findings and limitations | Final discussion and conclusion | Planned |


<a id="results-and-discussion"></a>
## 13. Results and Discussion

This section applies the three method blocks to the analysis-ready panel produced in
Section 8. Section 13.1 estimates the causal effect of CEO turnover on pay using a
DoWhy causal graph, a difference-in-differences regression, and a Sun & Abraham
event-study. Sections 13.2 and 13.3 apply supervised and unsupervised learning.

<a id="causal-inference-results"></a>
### 13.1 Causal Inference

We model CEO turnover as a governance shock in a causal DAG and use backdoor adjustment
to control for firm fundamentals (ROA, log assets, leverage, country, sector) that affect
both the probability of turnover and pay levels. Two estimators are reported:

- **DiD (TWFE):** `post_turnover` coefficient from a two-way fixed-effects regression
  with firm-level heterogeneity absorbed by `treated_firm`.
- **Event-study:** Sun & Abraham–style manual event-time dummies (τ = −3 … +3),
  omitting τ = −1 as the baseline, to test parallel pre-trends and trace the
  dynamic pay response.

In [ ]:
# ── 13.1 Causal Inference ────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="dowhy")
import networkx as nx
if not hasattr(nx.algorithms, "d_separated"):
    from networkx.algorithms.d_separation import is_d_separator
    nx.algorithms.d_separated = is_d_separator
from dowhy import CausalModel
import statsmodels.formula.api as smf

# Load the analysis panel saved in Section 8.4
df_a = pd.read_csv(DATA_DIR / "panel_analysis.csv")
df_a.loc[:, "event_time"] = df_a["event_time"].astype("Int64")

# Regression sample: rows with pay and at least one Compustat covariate
df_reg = df_a[
    df_a["log_w_real_pay"].notna() &
    df_a["log_assets"].notna() &
    df_a["roa"].notna()
].copy()

print(f"Regression sample : {len(df_reg):,} rows, {df_reg['isin'].nunique()} firms")
print(f"Treated (turnover): {df_reg['treated_firm'].sum()} rows  "
      f"({df_reg.groupby('isin')['treated_firm'].first().sum():.0f} firms)")
print(f"Year range        : {df_reg['year'].min()} \u2013 {df_reg['year'].max()}")

# ── 13.1.1  Causal DAG (DoWhy) ───────────────────────────────────────────────
causal_graph = """
digraph {
    ceo_turnover  -> log_w_real_pay;
    roa           -> ceo_turnover;
    roa           -> log_w_real_pay;
    log_assets    -> ceo_turnover;
    log_assets    -> log_w_real_pay;
    leverage      -> ceo_turnover;
    leverage      -> log_w_real_pay;
    hocountryname -> ceo_turnover;
    hocountryname -> log_w_real_pay;
    sector        -> ceo_turnover;
    sector        -> log_w_real_pay;
}
"""

model = CausalModel(
    data=df_reg,
    treatment="ceo_turnover",
    outcome="log_w_real_pay",
    graph=causal_graph,
)
model.view_model()

identified = model.identify_effect(proceed_when_unidentifiable=True)
adj_set = identified.get_backdoor_variables()
print(f"Backdoor adjustment set: {adj_set}")

# ── 13.1.2  DiD regression (operationalises backdoor adjustment) ──────────────
formula = (
    "log_w_real_pay ~ treated_firm + post_turnover "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_did = smf.ols(formula, data=df_reg).fit(cov_type="HC3")

print("\u2500\u2500 DiD: effect of post-turnover period on log CEO pay \u2500\u2500")
print(f"  post_turnover coef : {res_did.params['post_turnover']:.4f}")
print(f"  95% CI             : [{res_did.conf_int().loc['post_turnover', 0]:.4f}, "
      f"{res_did.conf_int().loc['post_turnover', 1]:.4f}]")
print(f"  p-value            : {res_did.pvalues['post_turnover']:.4f}")
print(f"  N                  : {int(res_did.nobs)},  R\u00b2 = {res_did.rsquared:.3f}")

In [ ]:
# ── 13.1.3  Event-study regression (Sun & Abraham style, manual) ─────────────
# Regress log pay on event-time dummies (\u03c4 = -3\u2026+3) for treated firms,
# omitting \u03c4 = -1 as the baseline. Control firms contribute to year FE.
# treated_firm absorbs the pre-period level difference between treated/control.
# Columns named et_m3\u2026et_p3 (\u201cm\u201d=minus, \u201cp\u201d=plus) to avoid patsy parsing
# hyphens in et_-3 as subtraction.

es_sample = df_reg[
    (df_reg["treated_firm"].eq(1) & df_reg["event_time"].between(-3, 3))
    | df_reg["treated_firm"].eq(0)
].copy()

def tau_col(t):
    return f"et_m{abs(t)}" if t < 0 else f"et_p{t}"

for tau in range(-3, 4):
    if tau == -1:
        continue
    es_sample[tau_col(tau)] = (
        (es_sample["treated_firm"].eq(1)) &
        (es_sample["event_time"] == tau)
    ).fillna(False).astype(int)

et_terms = " + ".join(tau_col(t) for t in range(-3, 4) if t != -1)
formula_es = (
    f"log_w_real_pay ~ {et_terms} + treated_firm "
    "+ roa + log_assets + leverage "
    "+ C(year) + C(hocountryname) + C(sector)"
)
res_es = smf.ols(formula_es, data=es_sample).fit(cov_type="HC3")

# Collect \u03c4 coefficients and CIs (\u03c4 = -1 is the omitted baseline \u2192 0)
taus_full = list(range(-3, 4))
coef_dict = {
    t: (0.0, 0.0, 0.0) if t == -1
    else (
        res_es.params.get(tau_col(t), np.nan),
        res_es.conf_int().loc[tau_col(t), 0] if tau_col(t) in res_es.params else np.nan,
        res_es.conf_int().loc[tau_col(t), 1] if tau_col(t) in res_es.params else np.nan,
    )
    for t in taus_full
}
coefs_full = [coef_dict[t][0] for t in taus_full]
ci_lo_full = [coef_dict[t][1] for t in taus_full]
ci_hi_full = [coef_dict[t][2] for t in taus_full]

fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(-0.5, color="red", linestyle="--", linewidth=1, label="Turnover (\u03c4=0)")
ax.fill_between(taus_full, ci_lo_full, ci_hi_full, alpha=0.2, color="steelblue")
ax.plot(taus_full, coefs_full, marker="o", color="steelblue", linewidth=1.8, markersize=6)
ax.set_xticks(taus_full)
ax.set_xlabel("Event time (years relative to CEO turnover)")
ax.set_ylabel("\u0394 log CEO real pay vs \u03c4 = \u22121")
ax.set_title("Event-Study: CEO Pay Around Turnover (baseline = year before turnover, HC3 CI)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Event-study sample: {len(es_sample):,} rows  "
      f"(treated in window: {es_sample[es_sample['treated_firm']==1].shape[0]},  "
      f"control rows: {(es_sample['treated_firm']==0).sum()})")

print("\u2500\u2500 Event-time coefficients (log pay vs baseline \u03c4=\u22121):")
for t, c, lo, hi in zip(taus_full, coefs_full, ci_lo_full, ci_hi_full):
    marker = "\u2190 baseline" if t == -1 else ""
    print(f"  \u03c4 = {t:+d}:  \u03b2 = {c:+.4f}  [{lo:+.4f}, {hi:+.4f}]  {marker}")

### 13.2 Supervised Learning

In [ ]:
# Supervised learning analysis

### 13.3 Unsupervised / Generative Learning

In [ ]:
# Unsupervised / generative analysis

### 13.4 Discussion & Conclusion *(complete for final submission)*


*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
